# Waveform Processing

Loading, analyzing, and visualizing LeCroy TRC waveforms

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks

try:
    import lecroyparser
except ImportError as exc:
    raise ImportError("lecroyparser is required. Install with `pip install lecroyparser`.") from exc

#try:
#    import matplotlib
#    matplotlib.use("Agg")
#except Exception:
#    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
 )
logger = logging.getLogger("waveform")

ImportError: lecroyparser is required. Install with `pip install lecroyparser`.

In [ ]:
@dataclass(slots=True)
class Waveform:
    path: Path
    time_s: np.ndarray
    voltage_v: np.ndarray
    metadata: Dict[str, object]

    @property
    def time_ns(self) -> np.ndarray:
        """Return waveform time axis in nanoseconds for downstream analysis."""
        return self.time_s * 1e9

    @property
    def sample_interval_s(self) -> float:
        if self.time_s.size < 2:
            return float("nan")
        return float(np.median(np.diff(self.time_s)))

    @property
    def metadata_safe(self) -> Dict[str, object]:
        return dict(self.metadata) if self.metadata else {}

In [ ]:
@dataclass(slots=True)
class WaveformSummary:
    path: Path
    peak_time_ns: float
    amplitude_v: float
    baseline_v: float
    baseline_std_v: float
    baseline_spike: bool
    saturated: bool
    saturation_reason: str
    post_peak_positive: bool
    post_peak_time_ns: float
    post_peak_height_v: float
    post_peak_count: int
    post_peak_times_ns: List[float]
    post_peak_heights_v: List[float]
    charges_vns: Dict[str, float]

    def to_dict(self) -> Dict[str, object]:
        record = {
            "path": str(self.path),
            "peak_time_ns": self.peak_time_ns,
            "amplitude_v": self.amplitude_v,
            "baseline_v": self.baseline_v,
            "baseline_std_v": self.baseline_std_v,
            "baseline_spike": self.baseline_spike,
            "saturated": self.saturated,
            "saturation_reason": self.saturation_reason,
            "post_peak_positive": self.post_peak_positive,
            "post_peak_time_ns": self.post_peak_time_ns,
            "post_peak_height_v": self.post_peak_height_v,
            "post_peak_count": self.post_peak_count,
            "post_peak_times_ns": self.post_peak_times_ns,
            "post_peak_heights_v": self.post_peak_heights_v,
        }
        for key, value in self.charges_vns.items():
            record[f"charge_{key}_v_s"] = value
        return record

In [ ]:
@dataclass(slots=True)
class RunConfig:
    base_waveform_dir: Path
    sample: str = "Sample2"
    run_type: str = "AmBe"
    waveform_pattern: str = "*.trc"
    baseline_window_ns: Tuple[float, float] = (-130.0, -30.0)
    charge_window_ns: Tuple[float, float] = (20.0, 50.0)
    multi_charge_window_ns: Tuple[float, float, float, float] = (20.0, 50.0, 80.0, 120.0)
    pulse_polarity: str = "negative"  # or 'positive'
    show_plots: bool = False
    max_waveforms: Optional[int] = 100
    
    ## saturation detection parameters
    amplitude_tolerance_v: float = 0.02  # minimum peak magnitude to evaluate saturation
    flatness_window_ns: float = 10.0  # window around the peak to inspect for plateaus
    flatness_tolerance_v: float = 0.0015  # allowable deviation from the peak value inside the window
    flatness_fraction: float = 0.3  # fraction of samples within tolerance needed to call a plateau
    flatness_min_consecutive: int = 5  # consecutive samples within tolerance required for clipping
    
    ## baseline spike detection parameters
    ## for AmBe 0.003 used for spike threshold
    baseline_spike_threshold_v: float = 0.005  # pulses above this in the baseline window trigger a flag
    baseline_spike_std_scale: float = 3.0  # also flag when deviation > scale * baseline_std
    baseline_spike_min_consecutive: int = 1  # minimum consecutive samples above threshold to call spike
    
    trigger_time_ns: float = 0.0  # oscilloscope trigger reference
    main_peak_window_ns: float = 10.0  # main peak should fall within +/- this window around trigger
    post_peak_search_ns: float = 50.0  # start looking for post-peaks at trigger + delay
    post_peak_threshold_v: float = 0.02  # minimum excursion above baseline for post-peak detection
  
    def __post_init__(self) -> None:
        polarity = self.pulse_polarity.lower()
        if polarity not in {"negative", "positive"}:
            raise ValueError("pulse_polarity must be 'negative' or 'positive'")
        self.pulse_polarity = polarity

    @property
    def waveform_dir(self) -> Path:
        return self.base_waveform_dir / self.sample / self.run_type

    @property
    def results_dir(self) -> Path:
        return Path("/Users/ukose/sw/Work/Neutron3D/Data/Analysis_results")

    @property
    def plots_dir(self) -> Path:
        return self.results_dir / "plots"

    def plot_path(self, suffix: str, ext: str = "png") -> Path:
        sample_part = self.sample.replace(" ", "_").lower()
        run_part = self.run_type.replace(" ", "_").lower()
        safe_suffix = suffix.replace(" ", "_").lower()
        return self.plots_dir / f"{sample_part}_{run_part}_{safe_suffix}.{ext}"

    def subset_path(self, suffix: str, ext: str = "pkl") -> Path:
        sample_part = self.sample.replace(" ", "_").lower()
        run_part = self.run_type.replace(" ", "_").lower()
        safe_suffix = suffix.replace(" ", "_").lower()
        return self.results_dir / f"{sample_part}_{run_part}_{safe_suffix}.{ext}"

    def _results_filename(self, ext: str) -> str:
        sample_part = self.sample.replace(" ", "_").lower()
        run_part = self.run_type.replace(" ", "_").lower()
        return f"{sample_part}_{run_part}_pulse_summary.{ext}"

    @property
    def results_csv_path(self) -> Path:
        return self.results_dir / self._results_filename("csv")

    @property
    def results_pkl_path(self) -> Path:
        return self.results_dir / self._results_filename("pkl")

    @property
    def charge_hist_path(self) -> Path:
        sample_part = self.sample.replace(" ", "_").lower()
        run_part = self.run_type.replace(" ", "_").lower()
        return self.results_dir / f"{sample_part}_{run_part}_charge_hist.png"

    @property
    def baseline_hist_path(self) -> Path:
        sample_part = self.sample.replace(" ", "_").lower()
        run_part = self.run_type.replace(" ", "_").lower()
        return self.results_dir / f"{sample_part}_{run_part}_baseline_hist.png"

    @property
    def polarity_sign(self) -> int:
        return -1 if self.pulse_polarity == "negative" else 1

In [ ]:
def find_trc_files(config: RunConfig) -> List[Path]:
    directory = config.waveform_dir
    if not directory.exists():
        logger.warning("Waveform directory %s does not exist", directory)
        return []
    files = sorted(directory.rglob(config.waveform_pattern))
    logger.info("Found %d TRC files under %s", len(files), directory)
    return files


def _extract_metadata(scope: object) -> Dict[str, object]:
    metadata: Dict[str, object] = {}
    for attr in ("instrumentName", "instrument_name", "instrument"):
        value = getattr(scope, attr, None)
        if value:
            metadata["instrument"] = value
            break
    metadata["record_length"] = len(getattr(scope, "y", []))
    metadata["sample_interval_s"] = getattr(scope, "dt", getattr(scope, "horizontal_interval", None))
    metadata["trigger_time"] = getattr(scope, "triggerTime", getattr(scope, "trigger_time", None))
    return metadata


def load_scope_data(path: Path) -> Optional[Waveform]:
    try:
        scope = lecroyparser.ScopeData(str(path), parseAll=False)
    except Exception as exc:
        logger.error("Failed to read %s: %s", path, exc, exc_info=True)
        return None

    time_s = np.asarray(scope.x, dtype=np.float64)
    voltage_v = np.asarray(scope.y, dtype=np.float64)
    metadata = _extract_metadata(scope)
    return Waveform(path=path, time_s=time_s, voltage_v=voltage_v, metadata=metadata)


def load_waveforms(paths: Sequence[Path]) -> Tuple[List[Waveform], List[Path]]:
    waveforms: List[Waveform] = []
    failed: List[Path] = []
    for path in paths:
        waveform = load_scope_data(path)
        if waveform is None:
            failed.append(path)
        else:
            waveforms.append(waveform)
    return waveforms, failed

In [ ]:
def locate_peak(time_ns: np.ndarray, voltage_v: np.ndarray, config: RunConfig) -> int:
    if voltage_v.size == 0:
        raise ValueError("Waveform contains no samples")
    window_start = config.trigger_time_ns - config.main_peak_window_ns
    window_end = config.trigger_time_ns + config.main_peak_window_ns
    mask = (time_ns >= window_start) & (time_ns <= window_end)
    if np.any(mask):
        subset = voltage_v[mask]
        candidate_indices = np.flatnonzero(mask)
        if config.polarity_sign < 0:
            local_idx = int(np.argmin(subset))
        else:
            local_idx = int(np.argmax(subset))
        return int(candidate_indices[local_idx])
    return int(np.argmin(voltage_v) if config.polarity_sign < 0 else np.argmax(voltage_v))


def compute_baseline(
    time_ns: np.ndarray, voltage_v: np.ndarray, peak_idx: int, config: RunConfig
) -> Tuple[float, float, bool]:
    """Compute baseline and detect spikes robustly.

    Spike detection uses a threshold that is the maximum of a fixed absolute threshold and
    a multiple of the measured baseline standard deviation. It optionally requires a minimum
    number of consecutive samples above that threshold to declare a spike.
    """
    peak_time = float(time_ns[peak_idx])
    window_start = peak_time + config.baseline_window_ns[0]
    window_end = peak_time + config.baseline_window_ns[1]
    mask = (time_ns >= window_start) & (time_ns <= window_end)
    if mask.sum() < 3:
        raise ValueError("Baseline window contains fewer than three samples")
    segment = voltage_v[mask]
    baseline = float(np.mean(segment))
    baseline_std = float(np.std(segment, ddof=0))
    # subtract the baseline from the segment before testing for spikes
    corrected_segment = segment - baseline

    # adaptive threshold: absolute OR k * baseline_std
    thresh = max(float(config.baseline_spike_threshold_v), float(config.baseline_spike_std_scale) * baseline_std)
    above = np.abs(corrected_segment) > thresh
    if config.baseline_spike_min_consecutive <= 1:
        has_spike = bool(np.any(above))
    else:
        has_spike = (_max_consecutive_true(above) >= int(config.baseline_spike_min_consecutive))
    return baseline, baseline_std, has_spike


def integrate_window(
    time_ns: np.ndarray, aligned_signal_v: np.ndarray, start_ns: float, end_ns: float
) -> float:
    mask = (time_ns >= start_ns) & (time_ns <= end_ns)
    if mask.sum() < 2:
        return 0.0
    return float(np.trapz(aligned_signal_v[mask], time_ns[mask]) * 1e-9)


def _max_consecutive_true(values: np.ndarray) -> int:
    best = count = 0
    for item in values:
        if item:
            count += 1
            best = max(best, count)
        else:
            count = 0
    return best


def detect_saturation(
    aligned_signal_v: np.ndarray, time_ns: np.ndarray, peak_idx: int, config: RunConfig
) -> Tuple[bool, str]:
    amplitude = float(aligned_signal_v[peak_idx])
    if amplitude < config.amplitude_tolerance_v:
        return False, ""
    sample_diffs = np.diff(time_ns)
    step_ns = float(np.median(sample_diffs)) if sample_diffs.size else 1.0
    if not np.isfinite(step_ns) or step_ns <= 0.0:
        step_ns = 1.0
    half_window_pts = max(1, int(np.ceil(config.flatness_window_ns / step_ns)))
    start_idx = max(0, peak_idx - half_window_pts)
    end_idx = min(aligned_signal_v.size, peak_idx + half_window_pts + 1)
    plateau_region = aligned_signal_v[start_idx:end_idx]
    if plateau_region.size == 0:
        return False, ""
    peak_value = aligned_signal_v[peak_idx]
    within_tol = np.abs(plateau_region - peak_value) <= config.flatness_tolerance_v
    fraction = float(np.mean(within_tol))
    longest_run = _max_consecutive_true(within_tol)
    if peak_idx <= 1 or peak_idx >= aligned_signal_v.size - 2:
        return True, "peak at acquisition boundary"
    if fraction >= config.flatness_fraction and longest_run >= config.flatness_min_consecutive:
        return True, "flat-top plateau"
    return False, ""

In [ ]:
def detect_post_peak_positive(
    corrected_v: np.ndarray, time_ns: np.ndarray, peak_idx: int, config: RunConfig
) -> Tuple[bool, float, float, int, List[float], List[float]]:
    start_ns = max(config.trigger_time_ns + config.post_peak_search_ns, float(time_ns[peak_idx]))
    mask = time_ns >= start_ns
    if mask.sum() < 3:
        return False, float("nan"), 0.0, 0, [], []

    post_segment = corrected_v[mask]
    post_time_ns = time_ns[mask]

    if config.polarity_sign < 0:
        search_segment = -post_segment
        sign = -1.0
    else:
        search_segment = post_segment
        sign = 1.0

    peaks, properties = find_peaks(search_segment, height=config.post_peak_threshold_v)
    if peaks.size == 0:
        return False, float("nan"), 0.0, 0, [], []

    peak_times_all = post_time_ns[peaks].astype(float).tolist()
    peak_heights_signed = (sign * properties["peak_heights"]).astype(float).tolist()
    best_idx = int(np.argmax(properties["peak_heights"]))
    primary_time = float(peak_times_all[best_idx])
    primary_height = float(peak_heights_signed[best_idx])
    return True, primary_time, primary_height, len(peak_times_all), peak_times_all, peak_heights_signed


def analyze_waveform(waveform: Waveform, config: RunConfig) -> WaveformSummary:
    time_ns = waveform.time_ns
    peak_idx = locate_peak(time_ns, waveform.voltage_v, config)
    baseline_v, baseline_std_v, baseline_spike = compute_baseline(
        time_ns, waveform.voltage_v, peak_idx, config
)
    corrected_v = waveform.voltage_v - baseline_v
    aligned_signal = corrected_v * config.polarity_sign
    pre_trigger_mask = time_ns <= (config.trigger_time_ns - config.main_peak_window_ns)
    if np.any(pre_trigger_mask):
        pre_segment = aligned_signal[pre_trigger_mask]
        pre_peaks, _ = find_peaks(pre_segment, height=config.post_peak_threshold_v)
        if pre_peaks.size > 0:
            baseline_spike = True
    peak_time_ns = float(time_ns[peak_idx])
    amplitude_v = float(aligned_signal[peak_idx])
    saturated, saturation_reason = detect_saturation(aligned_signal, time_ns, peak_idx, config)
    (
        post_peak_positive,
        post_peak_time_ns,
        post_peak_height_v,
        post_peak_count,
        post_peak_times_ns,
        post_peak_heights_v,
    ) = detect_post_peak_positive(corrected_v, time_ns, peak_idx, config)
    charge_pre_ns, charge_post_ns = config.charge_window_ns
    primary_start = peak_time_ns - charge_pre_ns
    primary_end = peak_time_ns + charge_post_ns
    charges: Dict[str, float] = {
        "primary": integrate_window(time_ns, aligned_signal, primary_start, primary_end)
    }
    if config.multi_charge_window_ns and len(config.multi_charge_window_ns) > 1:
        pre_ns = config.multi_charge_window_ns[0]
        for window_ns in config.multi_charge_window_ns[1:]:
            start_ns = peak_time_ns - pre_ns
            end_ns = peak_time_ns + window_ns
            label = f"multi_{int(window_ns)}" if float(window_ns).is_integer() else f"multi_{window_ns:g}"
            charges[label] = integrate_window(time_ns, aligned_signal, start_ns, end_ns)
    return WaveformSummary(
        path=waveform.path,
        peak_time_ns=peak_time_ns,
        amplitude_v=amplitude_v,
        baseline_v=baseline_v,
        baseline_std_v=baseline_std_v,
        baseline_spike=baseline_spike,
        saturated=saturated,
        saturation_reason=saturation_reason,
        post_peak_positive=post_peak_positive,
        post_peak_time_ns=post_peak_time_ns,
        post_peak_height_v=post_peak_height_v,
        post_peak_count=post_peak_count,
        post_peak_times_ns=post_peak_times_ns,
        post_peak_heights_v=post_peak_heights_v,
        charges_vns=charges,
    )


def analyze_waveforms_batch(
    waveforms: Sequence[Waveform], config: RunConfig
) -> Tuple[List[WaveformSummary], List[Tuple[Path, Exception]]]:
    summaries: List[WaveformSummary] = []
    failures: List[Tuple[Path, Exception]] = []
    for waveform in waveforms:
        try:
            summaries.append(analyze_waveform(waveform, config))
        except Exception as exc:
            logger.error("Analysis failed for %s: %s", waveform.path, exc, exc_info=True)
            failures.append((waveform.path, exc))
    return summaries, failures


def summaries_to_dataframe(summaries: Sequence[WaveformSummary]) -> pd.DataFrame:
    if not summaries:
        return pd.DataFrame()
    return pd.DataFrame([summary.to_dict() for summary in summaries])

In [ ]:
def plot_waveform_grid(
    waveforms: Sequence[Waveform],
    summaries: Sequence[WaveformSummary],
    config: RunConfig,
    *,
    limit: Optional[int] = 9,
    cols: int = 3,
    highlight_times: Optional[Sequence[Optional[float]]] = None,
    highlight_colors: Optional[Sequence[str]] = None,
    save_path: Optional[Path] = None,
 ) -> None:
    if not config.show_plots:
        return
    count = len(waveforms) if limit is None else min(limit, len(waveforms))
    if count == 0:
        return
    cols = max(1, cols)
    rows = int(np.ceil(count / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3), sharex=True)
    axes_arr = np.atleast_1d(axes).ravel()
    for idx in range(count):
        waveform = waveforms[idx]
        summary = summaries[idx]
        ax = axes_arr[idx]
        corrected = waveform.voltage_v - summary.baseline_v
        time_ns = waveform.time_ns
        ax.plot(time_ns, corrected, color="tab:blue", linewidth=1.0)
        ax.axvline(summary.peak_time_ns, color="tab:red", linestyle="--", linewidth=0.8)
        ax.axhline(0.0, color="black", linestyle=":", linewidth=0.7)
        start_primary = summary.peak_time_ns - config.charge_window_ns[0]
        end_primary = summary.peak_time_ns + config.charge_window_ns[1]
        ax.axvspan(start_primary, end_primary, color="tab:purple", alpha=0.15)
        if highlight_times and idx < len(highlight_times):
            t_ns = highlight_times[idx]
            if t_ns is not None and np.isfinite(t_ns):
                color = highlight_colors[idx] if highlight_colors and idx < len(highlight_colors) else "tab:orange"
                ax.axvline(float(t_ns), color=color, linestyle="-", linewidth=0.9)
        ax.set_title(waveform.path.name, fontsize=9)
        ax.grid(alpha=0.3)
    for ax in axes_arr[count:]:
        ax.axis("off")
    fig.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_waveform_stack(
    waveforms: Sequence[Waveform],
    summaries: Sequence[WaveformSummary],
    config: RunConfig,
    *,
    limit: Optional[int] = 25,
    alpha: float = 0.35,
    colors: Optional[Sequence[str]] = None,
    normalize: bool = False,
    title: Optional[str] = None,
    figsize: Tuple[int, int] = (12, 6),
    highlight_times: Optional[Sequence[Optional[float]]] = None,
    highlight_colors: Optional[Sequence[str]] = None,
    save_path: Optional[Path] = None,
 ) -> None:
    if not config.show_plots:
        return
    if limit is not None and len(waveforms) > limit:
        waveforms = waveforms[:limit]
        summaries = summaries[:limit]
        if highlight_times:
            highlight_times = highlight_times[:limit]
        if highlight_colors:
            highlight_colors = highlight_colors[:limit]
    if not waveforms:
        return
    if colors is None:
        cmap = plt.cm.get_cmap("viridis", len(waveforms))
        colors = [cmap(i) for i in range(len(waveforms))]
    plt.figure(figsize=figsize)
    for idx, (waveform, summary) in enumerate(zip(waveforms, summaries)):
        time_ns = waveform.time_ns
        trace = waveform.voltage_v - summary.baseline_v
        if normalize:
            max_abs = float(np.max(np.abs(trace))) if trace.size else 0.0
            if max_abs > 0:
                trace = trace / max_abs
        plt.plot(time_ns, trace, color=colors[idx], alpha=alpha, linewidth=1.0)
        ## do not show peak line
        #plt.axvline(summary.peak_time_ns, color=colors[idx], alpha=0.2, linestyle=":", linewidth=0.8)
        if highlight_times and idx < len(highlight_times):
            t_ns = highlight_times[idx]
            if t_ns is not None and np.isfinite(t_ns):
                color = highlight_colors[idx] if highlight_colors and idx < len(highlight_colors) else "tab:orange"
                plt.axvline(float(t_ns), color=color, linestyle="-", linewidth=0.9, alpha=0.8)
    charge_pre, charge_post = config.charge_window_ns
    ref_peak = summaries[0].peak_time_ns
    plt.axvspan(
        ref_peak - charge_pre,
        ref_peak + charge_post,
        color="tab:purple",
        alpha=0.12,
        label="Primary integration",
    )
    plt.axhline(0.0, color="black", linestyle=":", linewidth=0.8)
    plt.xlabel("Time [ns]")
    plt.ylabel("Voltage - baseline [V]")
    plt.title(title or f"Overlay of {len(waveforms)} waveforms")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.show()

In [ ]:
def plot_waveform(
    waveform: Waveform,
    summary: WaveformSummary,
    config: RunConfig,
    *,
    show: Optional[bool] = None,
    highlight_times: Optional[Sequence[float]] = None,
    highlight_colors: Optional[Sequence[str]] = None,
    save_path: Optional[Path] = None,
 ) -> None:
    display_plot = config.show_plots if show is None else show
    if not display_plot:
        return

    time_ns = waveform.time_ns
    baseline_v = summary.baseline_v
    corrected = waveform.voltage_v - baseline_v

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(time_ns, waveform.voltage_v, color="tab:blue", linewidth=1.2)
    axes[0].axhline(baseline_v, color="tab:orange", linestyle="--", linewidth=1)
    # show baseline window (relative to peak time)
    try:
        bw_start = summary.peak_time_ns + config.baseline_window_ns[0]
        bw_end = summary.peak_time_ns + config.baseline_window_ns[1]
        axes[0].axvspan(bw_start, bw_end, color="tab:orange", alpha=0.12, label="Baseline window")
    except Exception:
        # if config or summary lacks the fields, skip shading
        pass
    axes[0].set_ylabel("Voltage [V]")
    axes[0].set_title(f"Raw waveform: {waveform.path.name}")
    axes[0].grid(alpha=0.3)

    axes[1].plot(time_ns, corrected, color="tab:green", linewidth=1.2)
    axes[1].axvline(summary.peak_time_ns, color="tab:red", linestyle="--", linewidth=1, label="Peak time")
    # also indicate baseline window on the corrected trace for clarity
    try:
        bw_start = summary.peak_time_ns + config.baseline_window_ns[0]
        bw_end = summary.peak_time_ns + config.baseline_window_ns[1]
        axes[1].axvspan(bw_start, bw_end, color="tab:orange", alpha=0.08, label="Baseline window")
    except Exception:
        pass
    axes[1].axhline(0.0, color="black", linestyle=":", linewidth=0.8)
    axes[1].set_xlabel("Time [ns]")
    axes[1].set_ylabel("Voltage - baseline [V]")
    axes[1].grid(alpha=0.3)

    start_primary = summary.peak_time_ns - config.charge_window_ns[0]
    end_primary = summary.peak_time_ns + config.charge_window_ns[1]
    axes[1].axvspan(
        start_primary,
        end_primary,
        color="tab:purple",
        alpha=0.18,
        label="Primary integration",
    )

    if highlight_times:
        colors = list(highlight_colors or [])
        for idx, t_ns in enumerate(highlight_times):
            if not np.isfinite(t_ns):
                continue
            color = colors[idx] if idx < len(colors) else "tab:orange"
            axes[1].axvline(t_ns, color=color, linestyle="-", linewidth=1.0, alpha=0.8)

    if config.multi_charge_window_ns and len(config.multi_charge_window_ns) > 1:
        pre_ns = config.multi_charge_window_ns[0]
        post_windows = config.multi_charge_window_ns[1:]
        cmap = plt.cm.get_cmap("Blues", len(post_windows))
        for idx, window_ns in enumerate(post_windows):
            start_window = summary.peak_time_ns - pre_ns
            end_window = summary.peak_time_ns + window_ns
            if np.isclose(window_ns, config.charge_window_ns[1]):
                continue
            axes[1].axvspan(
                start_window,
                end_window,
                color=cmap(idx),
                alpha=0.12,
                label=f"Multi-window (+{window_ns:.0f} ns)",
            )

    handles, labels = axes[1].get_legend_handles_labels()
    if handles:
        seen = set()
        dedup_handles: List = []
        dedup_labels: List[str] = []
        for handle, label in zip(handles, labels):
            if label in seen:
                continue
            seen.add(label)
            dedup_handles.append(handle)
            dedup_labels.append(label)
        axes[1].legend(dedup_handles, dedup_labels, loc="best")

    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()

# in the following set sample and run_type 

In [ ]:
CONFIG = RunConfig(
    base_waveform_dir=Path("/Users/ukose/sw/Work/Neutron3D/Data/Waveforms"),
    sample="Sample2",
    ##run_type="AmBe",
    ##run_type="cs137",
    ##run_type="na22",
    ##run_type="co60",
    ##run_type="Am241",
    run_type="Am241_gamma",
    ##run_type="AmBe_20cmHDPE",
    pulse_polarity="negative",
    show_plots=True,
    max_waveforms=None,
    amplitude_tolerance_v=0.02,
    flatness_window_ns=6.0,
    flatness_tolerance_v=0.001,
    flatness_fraction=0.35,
    flatness_min_consecutive=10,
    baseline_spike_threshold_v=0.03, ## for Cs137
    post_peak_threshold_v=0.1,
 )

all_trc_files = find_trc_files(CONFIG)
selected_files = all_trc_files[: CONFIG.max_waveforms] if CONFIG.max_waveforms else all_trc_files

waveforms, file_read_failures = load_waveforms(selected_files)
summaries, analysis_failures = analyze_waveforms_batch(waveforms, CONFIG)
summary_df = summaries_to_dataframe(summaries)

if not summary_df.empty:
    summary_df.insert(0, "entry_index", np.arange(len(summary_df)))
    summary_df.insert(1, "file_name", summary_df["path"].apply(lambda p: Path(p).name))

print(f"Total TRC files discovered: {len(all_trc_files)}")
print(f"Waveforms successfully loaded: {len(waveforms)}")
print(f"Files failed to load: {len(file_read_failures)}")
print(f"Waveforms analyzed: {len(summaries)}")
print(f"Analysis failures: {len(analysis_failures)}")

if file_read_failures:
    logger.warning("First failed file: %s", file_read_failures[0])
if analysis_failures:
    logger.warning("First analysis failure: %s", analysis_failures[0][0])

if not summary_df.empty:
    CONFIG.results_dir.mkdir(parents=True, exist_ok=True)
    CONFIG.plots_dir.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(CONFIG.results_csv_path, index=False)
    summary_df.to_pickle(CONFIG.results_pkl_path)
    logger.info("Saved summary to %s", CONFIG.results_csv_path)
    logger.info("Saved summary pickle to %s", CONFIG.results_pkl_path)

summary_df.head()

In [ ]:
amplitude_cut_to_refine_saturation = None

if summary_df.empty:
    print("No waveforms analyzed. Adjust CONFIG.max_waveforms or verify input files.")
else:
    CONFIG.plots_dir.mkdir(parents=True, exist_ok=True)
    ## check for saturated waveforms
    print("#####################################")
    print("\n--- Saturation Analysis ---")
    print("Plotting saturated waveforms if any are detected.\n")
    saturated_indices = [idx for idx, summary in enumerate(summaries) if summary.saturated]
    if not saturated_indices:
        print("No saturated waveforms detected with current parameters.")
    else:
        print(f"Detected {len(saturated_indices)} saturated waveforms.")
        sat_records = summary_df.iloc[saturated_indices].copy()
        display(sat_records[["entry_index", "file_name", "amplitude_v", "saturation_reason"]].head(10))
        ###
        original_show = CONFIG.show_plots
        CONFIG.show_plots = True
        try:
            first_sat_idx = saturated_indices[0]
            plot_waveform(
                waveforms[first_sat_idx],
                summaries[first_sat_idx],
                CONFIG,
                save_path=CONFIG.plot_path("saturated_example"),
            )
            plot_waveform_grid(
                [waveforms[i] for i in saturated_indices],
                [summaries[i] for i in saturated_indices],
                CONFIG,
                limit=6,
                save_path=CONFIG.plot_path("saturated_grid"),
            )
        finally:
            CONFIG.show_plots = original_show
        ###
    # build amplitude cut mask if requested
    amplitude_cut_mask = pd.Series(True, index=summary_df.index)
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        amplitude_cut_mask = summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)

    # summary of baseline spike flags
    baseline_spike_count = int(summary_df["baseline_spike"].sum())
    total_waveforms = len(summary_df)
    print(
        f"Baseline spike flags: {baseline_spike_count} of {total_waveforms} "
        f"(threshold {CONFIG.baseline_spike_threshold_v:.4f} V)."
    )
    if baseline_spike_count:
        flagged_baselines = summary_df[summary_df["baseline_spike"]][
            ["entry_index", "file_name", "baseline_v", "baseline_std_v", "baseline_spike"]
        ]
        display(flagged_baselines.head(10))
        CONFIG.show_plots = True
        try:
            indices = flagged_baselines["entry_index"].astype(int).tolist()
            selected_waveforms = [waveforms[idx] for idx in indices]
            selected_summaries = [summaries[idx] for idx in indices]
            plot_waveform_grid(selected_waveforms, selected_summaries, CONFIG, limit=6, save_path=CONFIG.plot_path("noisy_candidates_grid"))
        finally:
            CONFIG.show_plots = False

    # build masks for subsets
    saturated_mask = summary_df["saturated"].fillna(False)
    noisy_mask = summary_df["baseline_spike"].fillna(False) & (~saturated_mask) & amplitude_cut_mask
    post_peak_mask = (
        summary_df["post_peak_positive"].fillna(False)
        & (~saturated_mask)
        & (~summary_df["baseline_spike"].fillna(False))
        & amplitude_cut_mask
    )
    clean_mask = (
        (~summary_df["baseline_spike"].fillna(False))
        & (~saturated_mask)
        & (~summary_df["post_peak_positive"].fillna(False))
        & amplitude_cut_mask
    )

    print(f"Counts: saturated={int(saturated_mask.sum())}, noisy={int(noisy_mask.sum())}, post_peak={int(post_peak_mask.sum())}, clean={int(clean_mask.sum())}")

    # Plot baseline level and baseline noise histograms
    try:
        baseline_levels = summary_df["baseline_v"].dropna()
        baseline_noise = summary_df["baseline_std_v"].dropna()

        if baseline_levels.size:
            plt.figure(figsize=(6, 4))
            plt.hist(baseline_levels, bins=60, color="tab:orange", alpha=0.85)
            plt.xlabel("Baseline level [V]")
            plt.ylabel("Count")
            plt.title("Baseline level distribution")
            plt.grid(alpha=0.3)
            p = CONFIG.plot_path("baseline_level_hist")
            p.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(p, dpi=150)
            print(f"Saved baseline level histogram to {p}")
            plt.show()
        else:
            print("No baseline level values available to plot.")

        if baseline_noise.size:
            plt.figure(figsize=(6, 4))
            plt.hist(baseline_noise, bins=60, color="tab:purple", alpha=0.85)
            plt.xlabel("Baseline noise (std) [V]")
            plt.ylabel("Count")
            plt.title("Baseline noise (std) distribution")
            plt.grid(alpha=0.3)
            p2 = CONFIG.plot_path("baseline_noise_hist")
            p2.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(p2, dpi=150)
            print(f"Saved baseline noise histogram to {p2}")
            plt.show()
        else:
            print("No baseline noise values available to plot.")
    except Exception as e:
        print("Failed to plot baseline histograms:", e)

    # preserve downstream variables (no change)
    baseline_df = summary_df[["entry_index", "file_name", "baseline_v", "baseline_std_v", "baseline_spike"]].copy()
    noisy_candidates = summary_df[noisy_mask].copy()
    noisy_indices = list(noisy_candidates["entry_index"].values)
    noisy_waveforms = [waveforms[int(i)] for i in noisy_indices if 0 <= int(i) < len(waveforms)]
    noisy_summaries = [summaries[int(i)] for i in noisy_indices if 0 <= int(i) < len(summaries)]
    print(f"Prepared {len(noisy_candidates)} noisy baseline candidates (examples shown above if any).")


    print("Inspecting waveforms with peaks beyond main peak.\n")
    post_peak_flags = summary_df[post_peak_mask].copy()

    post_peak_count = len(post_peak_flags)
    threshold_v = CONFIG.post_peak_threshold_v
    print(
        f"Post-peak positives (>{threshold_v:.4f} V starting from {CONFIG.post_peak_search_ns:.0f} ns) detected in {post_peak_count} waveforms.",
    )
    if post_peak_count:
        display(
            post_peak_flags[[
                "entry_index",
                "file_name",
                "post_peak_time_ns",
                "post_peak_height_v",
                "amplitude_v",
                "baseline_std_v",
            ]].head(10)
        )
        original_show = CONFIG.show_plots
        CONFIG.show_plots = True
        try:
            first_post_idx = int(post_peak_flags.iloc[0]["entry_index"])
            plot_waveform(waveforms[first_post_idx], summaries[first_post_idx], CONFIG, save_path=CONFIG.plot_path("first_post_peak_waveform"))

            post_indices = post_peak_flags["entry_index"].astype(int).tolist()
            if post_indices:
                plot_waveform_grid(
                    [waveforms[i] for i in post_indices],
                    [summaries[i] for i in post_indices],
                    CONFIG,
                    limit=6,
                    save_path=CONFIG.plot_path("post_peak_grid"),
                )
        finally:
            CONFIG.show_plots = original_show
   


In [ ]:
CONFIG.plots_dir.mkdir(parents=True, exist_ok=True)
original_show = CONFIG.show_plots
CONFIG.show_plots = True
try:
    plot_waveform(
        waveforms[16],
        summaries[16],
        CONFIG,
        save_path=CONFIG.plot_path("manual_waveform_16"),
    )
finally:
    CONFIG.show_plots = original_show

In [ ]:
if summary_df.empty:
    print("No waveforms analyzed, so no overlay plots are available.")
else:
    CONFIG.plots_dir.mkdir(parents=True, exist_ok=True)

    saturated_indices = summary_df[summary_df["saturated"]]["entry_index"].astype(int).tolist()

    ## also aply further cut on amplitude if defined
    if amplitude_cut_to_refine_saturation is not None:
        print(f"Applying amplitude cut at {amplitude_cut_to_refine_saturation:.4f} V for further categorization.")
        ## baseline spike waveforms (excluding saturated)
        noisy_indices = summary_df[
            summary_df["baseline_spike"]
            & (~summary_df["saturated"])
            & (summary_df["amplitude_v"] < amplitude_cut_to_refine_saturation)
        ]["entry_index"].astype(int).tolist()
        print("Noisy indices count after amplitude cut: ", len(noisy_indices))
        ## post-peak positive waveforms (excluding saturated and baseline spike)
        post_peak_indices = summary_df[
            (summary_df["post_peak_positive"]) 
            & (~summary_df["saturated"]) 
            & (~summary_df["baseline_spike"])
            & (summary_df["amplitude_v"] < amplitude_cut_to_refine_saturation)
        ]["entry_index"].astype(int).tolist()
        print("Post-peak indices count after amplitude cut: ", len(post_peak_indices))
        ## clean waveforms (no baseline spike, no saturation, no post-peak positive)
        clean_indices = summary_df[
            (~summary_df["baseline_spike"]) 
            & (~summary_df["saturated"]) 
            & (~summary_df["post_peak_positive"])
            & (summary_df["amplitude_v"] < amplitude_cut_to_refine_saturation)
        ]["entry_index"].astype(int).tolist()
        print("Clean indices count after amplitude cut: ", len(clean_indices))
    else:
        noisy_indices = summary_df[summary_df["baseline_spike"]& (~summary_df["saturated"])]["entry_index"].astype(int).tolist()
       ## post-peak positive waveforms (excluding saturated and baseline spike)
    
        post_peak_indices = summary_df[
            (summary_df["post_peak_positive"]) & (~summary_df["saturated"])& (~summary_df["baseline_spike"])
        ]["entry_index"].astype(int).tolist()
        ## clean waveforms (no baseline spike, no saturation, no post-peak positive)
        clean_indices = summary_df[
            (~summary_df["baseline_spike"]) & (~summary_df["saturated"]) & (~summary_df["post_peak_positive"])
        ]["entry_index"].astype(int).tolist()
    
    
    original_show = CONFIG.show_plots
    CONFIG.show_plots = True
    try:
        if saturated_indices:
            saturated_waveforms = [waveforms[idx] for idx in saturated_indices]
            saturated_summaries = [summaries[idx] for idx in saturated_indices]
            plot_waveform_stack(
                saturated_waveforms,
                saturated_summaries,
                CONFIG,
                limit=None,
                normalize=False,
                title="Overlay of saturated waveforms",
                save_path=CONFIG.plot_path("saturated_stack"),
            )
        else:
            print("No saturated waveforms present in current analysis.")

        if noisy_indices:
            noisy_waveforms = [waveforms[idx] for idx in noisy_indices]
            noisy_summaries = [summaries[idx] for idx in noisy_indices]
            plot_waveform_stack(
                noisy_waveforms,
                noisy_summaries,
                CONFIG,
                limit=None,
                normalize=False,
                title="Overlay of baseline-spike waveforms",
                save_path=CONFIG.plot_path("baseline_spike_stack"),
            )
        else:
            print("No baseline spikes flagged with the current thresholds.")
        
        if post_peak_indices:
            post_waveforms = [waveforms[idx] for idx in post_peak_indices]
            post_summaries = [summaries[idx] for idx in post_peak_indices]
            highlight_times = [
                summaries[idx].post_peak_time_ns if np.isfinite(summaries[idx].post_peak_time_ns) else None
                for idx in post_peak_indices
            ]
            highlight_colors = [
                "tab:red"
                if (summaries[idx].post_peak_heights_v[0] if summaries[idx].post_peak_heights_v else summaries[idx].post_peak_height_v) > 0
                else "tab:orange"
                for idx in post_peak_indices
            ]
            plot_waveform_stack(
                post_waveforms,
                post_summaries,
                CONFIG,
                limit=None,
                normalize=False,
                title="Overlay of post-peak excursions (non-saturated, no baseline spikes)",
                #highlight_times=highlight_times,
                #highlight_colors=highlight_colors,
                save_path=CONFIG.plot_path("post_peak_non_saturated_stack"),
            )
        else:
            print("No post-peak excursions identified after excluding saturated waveforms.")
        if clean_indices:
            clean_waveforms = [waveforms[idx] for idx in clean_indices]
            clean_summaries = [summaries[idx] for idx in clean_indices]
            plot_waveform_stack(
                clean_waveforms,
                clean_summaries,
                CONFIG,
                limit=2000,
                normalize=False,
                title="Overlay of clean waveforms (no spikes, no saturation, no post-peaks)",
                save_path=CONFIG.plot_path("clean_waveforms_stack"),
            )
            if len(clean_waveforms) > 2000:
                print("Limited clean waveform overlay to 2000 traces for responsiveness.")
        else:
            print("No clean waveforms available under current filters.")
    finally:
        CONFIG.show_plots = original_show

In [ ]:
if summary_df.empty:
    print("No waveforms analyzed, so no subset exports were created.")
else:
    CONFIG.results_dir.mkdir(parents=True, exist_ok=True)
    if "entry_index" not in summary_df.columns:
        summary_df.insert(0, "entry_index", np.arange(len(summary_df)))
    bool_columns = ["saturated", "baseline_spike", "post_peak_positive"]
    for column in bool_columns:
        if column in summary_df.columns:
            summary_df[column] = summary_df[column].fillna(False).astype(bool)
        else:
            summary_df[column] = False
    amplitude_cut_mask = pd.Series(True, index=summary_df.index)
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        amplitude_cut_mask = summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
    saturated_mask = summary_df["saturated"]
    noisy_mask = summary_df["baseline_spike"] & (~summary_df["saturated"]) & amplitude_cut_mask
    post_peak_mask = (
        summary_df["post_peak_positive"]
        & (~summary_df["saturated"])
        & (~summary_df["baseline_spike"])
        & amplitude_cut_mask
    )
    clean_mask = (
        (~summary_df["baseline_spike"])
        & (~summary_df["saturated"])
        & (~summary_df["post_peak_positive"])
        & amplitude_cut_mask
    )
    subsets = {
        "saturated_waveforms": summary_df[saturated_mask].copy(),
        "noisy_waveforms": summary_df[noisy_mask].copy(),
        "post_peak_waveforms": summary_df[post_peak_mask].copy(),
        "clean_waveforms": summary_df[clean_mask].copy(),
    }
    summary_records = []
    for label, subset in subsets.items():
        path = CONFIG.subset_path(label)
        subset.to_pickle(path)
        summary_records.append({"subset": label, "count": len(subset), "path": str(path)})
        print(f"Saved {len(subset):4d} rows to {path}")
    subset_overview = pd.DataFrame(summary_records)
    display(subset_overview)

In [ ]:
if summary_df.empty:
    print("No waveforms analyzed, so no charge distributions are available.")
else:
    print("\n--- Charge Distribution Comparison: Clean vs Post-peak ---\n")
    if "entry_index" not in summary_df.columns:
        summary_df.insert(0, "entry_index", np.arange(len(summary_df)))
    bool_columns = ["saturated", "baseline_spike", "post_peak_positive"]
    for column in bool_columns:
        if column in summary_df.columns:
            summary_df[column] = summary_df[column].fillna(False).astype(bool)
        else:
            summary_df[column] = False
    charge_columns = [column for column in summary_df.columns if column.startswith("charge_")]
    if not charge_columns:
        print("Charge columns missing; rerun analysis with charge calculations enabled.")
    else:
        charge_column = "charge_primary_v_s" if "charge_primary_v_s" in charge_columns else charge_columns[0]
        if charge_column != "charge_primary_v_s":
            print(f"Using {charge_column} for charge plots (primary charge column not found).")
        amplitude_mask = pd.Series(True, index=summary_df.index)
        if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
            amplitude_mask = summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
            print(
                f"Applied amplitude mask < {float(amplitude_cut_to_refine_saturation):.4f} V for clean/post-peak comparison.",
            )
        saturated_mask = summary_df["saturated"]
        baseline_mask = summary_df["baseline_spike"]
        post_peak_mask = summary_df["post_peak_positive"] & (~saturated_mask) & (~baseline_mask) & amplitude_mask
        clean_mask = (~saturated_mask) & (~baseline_mask) & (~summary_df["post_peak_positive"]) & amplitude_mask
        clean_charge = summary_df.loc[clean_mask, charge_column].dropna()
        post_peak_charge = summary_df.loc[post_peak_mask, charge_column].dropna()
        print(f"Clean subset count: {len(clean_charge)}")
        print(f"Post-peak subset count: {len(post_peak_charge)}")
        if clean_charge.empty and post_peak_charge.empty:
            print("Both clean and post-peak subsets are empty after filtering; skipping plot.")
        else:
            combined = pd.concat([clean_charge, post_peak_charge]) if not clean_charge.empty and not post_peak_charge.empty else (clean_charge if not clean_charge.empty else post_peak_charge)
            if combined.empty or combined.min() == combined.max():
                print("Insufficient spread in charge data to build comparison histogram.")
            else:
                bins = 80
                edges = np.linspace(float(combined.min()), float(combined.max()), bins + 1)
                plt.figure(figsize=(8, 4))
                if not clean_charge.empty:
                    plt.hist(
                        clean_charge,
                        bins=edges,
                        histtype="step",
                        linewidth=1.8,
                        color="tab:green",
                        label=f"Clean (n={len(clean_charge)})",
                    )
                if not post_peak_charge.empty:
                    plt.hist(
                        post_peak_charge,
                        bins=edges,
                        histtype="step",
                        linewidth=1.8,
                        color="tab:orange",
                        label=f"Post-peak (n={len(post_peak_charge)})",
                    )
                plt.xlabel("Charge [V·s]")
                plt.ylabel("Count")
                plt.title("Charge distribution comparison: Clean vs Post-peak")
                plt.grid(alpha=0.3)
                plt.legend()
                plt.tight_layout()
                plt.show()

In [ ]:
if summary_df.empty:
    print("No waveforms analyzed; average clean waveform unavailable.")
else:
    clean_mask = (
        (~summary_df.get("saturated", False))
        & (~summary_df.get("baseline_spike", False))
        & (~summary_df.get("post_peak_positive", False))
    )
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        clean_mask &= summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
    clean_indices = summary_df.index[clean_mask].tolist()
    if not clean_indices:
        print("No clean waveforms satisfy current filters; skipping average plot.")
    else:
        # Use the first clean trace as the reference time axis
        reference_idx = clean_indices[0]
        reference_entry = summary_df.loc[reference_idx]
        reference_entry_index = int(reference_entry["entry_index"]) if "entry_index" in reference_entry else int(reference_idx)
        ref_waveform = waveforms[reference_entry_index]
        ref_summary = summaries[reference_entry_index]
        ref_time_ns = ref_waveform.time_ns
        ref_trace = ref_waveform.voltage_v - ref_summary.baseline_v
        stack: List[np.ndarray] = [ref_trace]
        interpolated_count = 0
        skipped_count = 0
        for idx in clean_indices[1:]:
            entry = summary_df.loc[idx]
            entry_index = int(entry["entry_index"]) if "entry_index" in entry else int(idx)
            if entry_index < 0 or entry_index >= len(waveforms):
                skipped_count += 1
                continue
            wf = waveforms[entry_index]
            summary = summaries[entry_index]
            trace = wf.voltage_v - summary.baseline_v
            # identical time axis -> append raw trace
            if np.array_equal(wf.time_ns, ref_time_ns):
                stack.append(trace)
                continue
            # if time axis looks unusable, skip
            if wf.time_ns.size < 2 or not np.all(np.diff(wf.time_ns) > 0):
                skipped_count += 1
                continue
            # interpolate onto reference grid (allows partial overlap resulting in NaNs where no data)
            interpolated = np.interp(ref_time_ns, wf.time_ns, trace, left=np.nan, right=np.nan)
            stack.append(interpolated)
            interpolated_count += 1
        if not stack:
            print("No waveforms with compatible sampling found for averaging.")
        else:
            stack_arr = np.vstack(stack)
            valid_counts = np.sum(np.isfinite(stack_arr), axis=0)
            average_trace = np.nanmean(stack_arr, axis=0)
            valid_mask = valid_counts > 0
            if not np.any(valid_mask):
                print("Averaging failed; no overlapping time samples across clean waveforms.")
            else:
                plt.figure(figsize=(10, 4))
                plt.plot(ref_time_ns[valid_mask], average_trace[valid_mask], color="tab:blue", linewidth=1.2, label="Average clean waveform")
                plt.axhline(0.0, color="black", linestyle=":", linewidth=0.7)
                plt.xlabel("Time [ns]")
                plt.ylabel("Voltage - baseline [V]")
                plt.title(f"Average clean waveform ({valid_counts.max()} contributing traces at peak bin)")
                plt.grid(alpha=0.3)
                plt.tight_layout()
                avg_plot_path = CONFIG.plot_path("clean_waveform_average")
                avg_plot_path.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(avg_plot_path, dpi=150)
                plt.legend()
                plt.show()
                if interpolated_count or skipped_count:
                    print(f"Interpolated {interpolated_count} traces; skipped {skipped_count} traces during averaging.")
                print(f"Saved average clean waveform plot to {avg_plot_path}")

In [ ]:
# Post-peak detailed analysis: amplitude and charge (integration -10 ns / +20 ns around post-peak)
if summary_df.empty:
    print("No summaries available — run the analysis driver (Cell 7) first.")
else:
    # build candidate mask: post-peak positive, not saturated, not baseline spike
    mask = summary_df.get("post_peak_positive", False) & (~summary_df.get("saturated", False)) & (~summary_df.get("baseline_spike", False))
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        mask &= summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
    candidates = summary_df[mask].copy()
    if candidates.empty:
        print("No post-peak candidates found under current filters.")
    else:
        records = []
        for _, row in candidates.iterrows():
            entry_index = int(row.get("entry_index", row.name))
            if entry_index < 0 or entry_index >= len(waveforms):
                continue
            wf = waveforms[entry_index]
            summ = summaries[entry_index]
            time_ns = wf.time_ns
            # baseline-corrected, polarity-aligned trace
            aligned = (wf.voltage_v - summ.baseline_v) * CONFIG.polarity_sign
            main_amp = float(summ.amplitude_v) if getattr(summ, "amplitude_v", None) is not None else float(np.nan)
            main_time = float(getattr(summ, 'peak_time_ns', float('nan')))
            # determine post-peak time and height
            post_time = float(summ.post_peak_time_ns) if np.isfinite(getattr(summ, "post_peak_time_ns", float("nan"))) else (float(summ.post_peak_times_ns[0]) if summ.post_peak_times_ns else float("nan"))
            post_height = float(summ.post_peak_height_v) if getattr(summ, "post_peak_height_v", None) is not None and np.isfinite(getattr(summ, "post_peak_height_v", float("nan"))) else float(np.nan)
            if not np.isfinite(post_height) and np.isfinite(post_time):
                # sample nearest
                idx = int(np.argmin(np.abs(time_ns - post_time))) if time_ns.size else None
                if idx is not None and 0 <= idx < len(aligned):
                    post_height = float(aligned[idx])
            main_charge = float(summ.charges_vns.get("primary", np.nan)) if getattr(summ, "charges_vns", None) is not None else float(np.nan)
            post_charge = float("nan")
            if np.isfinite(post_time):
                start_ns = post_time - 10.0
                end_ns = post_time + 20.0
                try:
                    post_charge = integrate_window(time_ns, aligned, start_ns, end_ns)
                except Exception:
                    post_charge = float("nan")
            records.append({
                "entry_index": entry_index,
                "file_name": row.get("file_name", ""),
                "main_amplitude_v": main_amp,
                "main_peak_time_ns": main_time,
                "post_amplitude_v": post_height,
                "main_charge_v_s": main_charge,
                "post_charge_v_s": post_charge,
                "post_peak_time_ns": post_time,
            })

        post_df = pd.DataFrame(records)
        if post_df.empty:
            print("No valid post-peak records were collected.")
        else:
            # save table
            out_csv = CONFIG.results_dir / f"{CONFIG.sample.replace(' ','_')}_{CONFIG.run_type.replace(' ','_')}_post_peak_summary.csv"
            out_csv.parent.mkdir(parents=True, exist_ok=True)
            post_df.to_csv(out_csv, index=False)
            print(f"Saved post-peak summary to {out_csv}")
            # compute delta time (post - main) in ns
            post_df["delta_time_ns"] = post_df["post_peak_time_ns"] - post_df["main_peak_time_ns"]
            # save augmented table
            out_csv2 = CONFIG.results_dir / f"{CONFIG.sample.replace(' ','_')}_{CONFIG.run_type.replace(' ','_')}_post_peak_summary_with_delta.csv"
            post_df.to_csv(out_csv2, index=False)
            print(f"Saved post-peak summary with delta to {out_csv2}")
            # 1) amplitude main vs post scatter
            plt.figure(figsize=(6,5))
            plt.scatter(post_df["main_amplitude_v"], post_df["post_amplitude_v"], s=24, alpha=0.7, color='tab:blue')
            mins = np.nanmin(post_df[["main_amplitude_v","post_amplitude_v"]].values)
            maxs = np.nanmax(post_df[["main_amplitude_v","post_amplitude_v"]].values)
            if np.isfinite(mins) and np.isfinite(maxs):
                plt.plot([mins, maxs],[mins, maxs], linestyle='--', color='gray')
            plt.xlabel('Main amplitude [V]')
            plt.ylabel('Post-peak amplitude [V]')
            plt.title('Main vs Post-peak amplitude (clean post-peak candidates)')
            plt.grid(alpha=0.3)
            try:
                ppath = CONFIG.plot_path('amplitude_main_vs_post_detailed')
                ppath.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(ppath, dpi=150)
                print(f"Saved amplitude scatter to {ppath}")
            except Exception as e:
                print('Failed saving amplitude scatter:', e)
            plt.show()
            # 2) main vs post charge
            good = post_df[post_df["main_charge_v_s"].notna() & post_df["post_charge_v_s"].notna()]
            if not good.empty:
                plt.figure(figsize=(6,5))
                plt.scatter(good["main_charge_v_s"], good["post_charge_v_s"], s=24, alpha=0.7, color='tab:green')
                plt.plot([good["main_charge_v_s"].min(), good["main_charge_v_s"].max()],[good["main_charge_v_s"].min(), good["main_charge_v_s"].max()], linestyle='--', color='gray')
                plt.xlabel('Main charge [V·s]')
                plt.ylabel('Post-peak charge [V·s]')
                plt.title('Main vs Post-peak charge (-10/+20 ns integration)')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('charge_main_vs_post_detailed')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved charge scatter to {ppath}")
                except Exception as e:
                    print('Failed saving charge scatter:', e)
                plt.show()
            else:
                print('Insufficient finite charges for main vs post scatter.')
            # 3) histograms of charges
            plt.figure(figsize=(8,4))
            if post_df["main_charge_v_s"].notna().any():
                plt.hist(post_df["main_charge_v_s"].dropna(), bins=60, alpha=0.6, label=f"Main (n={post_df['main_charge_v_s'].notna().sum()})", color='tab:green')
            if post_df["post_charge_v_s"].notna().any():
                plt.hist(post_df["post_charge_v_s"].dropna(), bins=60, alpha=0.6, label=f"Post (n={post_df['post_charge_v_s'].notna().sum()})", color='tab:orange')
            plt.xlabel('Charge [V·s]')
            plt.ylabel('Count')
            plt.title('Charge distributions: main vs post-peak (integrated -10/+20 ns)')
            plt.legend()
            plt.grid(alpha=0.3)
            try:
                ppath = CONFIG.plot_path('charge_hist_main_post_detailed')
                ppath.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(ppath, dpi=150)
                print(f"Saved charge histogram to {ppath}")
            except Exception as e:
                print('Failed saving charge histogram:', e)
            plt.show()
            # 4) delta time plots
            if "delta_time_ns" in post_df.columns and post_df["delta_time_ns"].notna().any():
                d = post_df["delta_time_ns"].dropna()
                plt.figure(figsize=(6,4))
                plt.hist(d, bins=60, color='tab:purple', alpha=0.8)
                plt.xlabel('Post - Main time [ns]')
                plt.ylabel('Count')
                plt.title('Delta time distribution (post - main)')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('delta_time_hist')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved delta time histogram to {ppath}")
                except Exception as e:
                    print('Failed saving delta time histogram:', e)
                plt.show()
                # scatter delta vs post amplitude
                plt.figure(figsize=(6,4))
                plt.scatter(post_df['delta_time_ns'], post_df['post_amplitude_v'], s=20, alpha=0.7, color='tab:orange')
                plt.xlabel('Delta time [ns]')
                plt.ylabel('Post-peak amplitude [V]')
                plt.title('Delta time vs post-peak amplitude')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('delta_vs_post_amplitude')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved delta vs amplitude scatter to {ppath}")
                except Exception as e:
                    print('Failed saving delta vs amplitude scatter:', e)
                plt.show()
            else:
                print('No delta_time values available to plot.')

In [ ]:
# Plot a few post-peak waveforms with integration windows (-10 ns / +20 ns around post peak)
if summary_df.empty:
    print("No summaries available; run analysis (Cell 7) first.")
else:
    mask = summary_df.get("post_peak_positive", False) & (~summary_df.get("saturated", False)) & (~summary_df.get("baseline_spike", False))
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        mask &= summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
    candidates = summary_df[mask].copy()
    if candidates.empty:
        print("No post-peak candidates found under current filters.")
    else:
        n_show = min(6, len(candidates))
        selected = candidates.head(n_show)
        cols = 3
        rows = int(np.ceil(n_show / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3), sharex=True)
        axes = np.atleast_1d(axes).ravel()
        for ax_idx, (_, row) in enumerate(selected.iterrows()):
            ax = axes[ax_idx]
            entry_index = int(row.get("entry_index", row.name))
            if entry_index < 0 or entry_index >= len(waveforms):
                ax.text(0.5, 0.5, 'Index out of range', ha='center')
                ax.axis('off')
                continue
            wf = waveforms[entry_index]
            summ = summaries[entry_index]
            time_ns = wf.time_ns
            aligned = (wf.voltage_v - summ.baseline_v) * CONFIG.polarity_sign
            ax.plot(time_ns, aligned, color='tab:blue', linewidth=1.0)
            # main and post peak markers
            main_t = getattr(summ, 'peak_time_ns', float('nan'))
            post_t = getattr(summ, 'post_peak_time_ns', float('nan'))
            if not np.isfinite(main_t):
                main_t = float('nan')
            if not np.isfinite(post_t):
                # fallback to first post_peak_times_ns if present
                post_list = getattr(summ, 'post_peak_times_ns', [])
                post_t = float(post_list[0]) if post_list else float('nan')
            if np.isfinite(main_t):
                ax.axvline(main_t, color='tab:red', linestyle='--', linewidth=0.9, label='Main peak')
            if np.isfinite(post_t):
                ax.axvline(post_t, color='tab:orange', linestyle='-', linewidth=0.9, label='Post peak')
                # integration window around post peak: -10ns to +20ns
                start_ns = post_t - 10.0
                end_ns = post_t + 20.0
                ax.axvspan(start_ns, end_ns, color='tab:purple', alpha=0.15)
            ax.set_title(f"{Path(summ.path).name} (idx={entry_index})", fontsize=9)
            ax.set_xlabel('Time [ns]')
            ax.set_ylabel('Voltage - baseline [V]')
            ax.grid(alpha=0.25)
            # legend only on first subplot
            if ax_idx == 0:
                ax.legend(loc='best', fontsize=8)
        # turn off unused axes
        for ax in axes[n_show:]:
            ax.axis('off')
        fig.tight_layout()
        try:
            save_path = CONFIG.plot_path('post_peak_examples')
            save_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=150)
            print(f"Saved post-peak example figure to {save_path}")
        except Exception as exc:
            print(f"Failed to save post-peak examples figure: {exc}")
        plt.show()

# Code Walkthrough
This document summarizes the purpose of each part of the notebook so you can quickly recall what every block does when you revisit it later.

## Cell 1 – Title
Provides a short title so future readers know the notebook focuses on waveform processing.

## Cell 2 – Imports and Setup
* Imports standard libraries (logging, dataclasses, pathlib, typing) and scientific stack (matplotlib, numpy, pandas, scipy).
* Tries to import `lecroyparser`; if it is missing, raises an informative error.
* Configures a module-level logger named `waveform` so later helper functions can report progress and errors.

## Cell 3 – Data Classes and Paths
* `RunConfig` centralises every configurable knob, such as directories, pulse polarity, baseline/charge windows, and saturation heuristics.
    * `waveform_dir` builds the TRC search directory from `base_waveform_dir / sample / run_type`.
    * `results_dir` points to `Data/Analysis_results` so all derived products land in a shared location.
    * `_results_filename` assembles filenames containing the sample and run name, producing CSV/PKL paths via `results_csv_path` and `results_pkl_path`.
    * `charge_hist_path` gives the destination for the saved primary-charge histogram.
* `Waveform` stores raw waveform arrays with metadata and exposes `time_ns` for convenience.
* `WaveformSummary` stores per-trace metrics, including charges and saturation flags, and converts itself to a flat dict for DataFrame construction.

## Cell 4 – File Discovery and Loading
* `find_trc_files` searches the configured directory for `.trc` files and logs what it finds.
* `_extract_metadata` pulls basic instrument and sampling meta-information from the `lecroyparser` object.
* `load_scope_data` reads a TRC file into numpy arrays, returning a `Waveform`; failures are logged for debugging.
* `load_waveforms` loops across selected paths, collecting successful loads and tracking any problematic files separately.

## Cell 5 – Waveform Analysis Helpers
* `compute_baseline` averages the signal inside the requested baseline window and raises an error if the window is outside the trace.
* `locate_peak` chooses the minimum or maximum sample depending on pulse polarity.
* `integrate_window` integrates voltage over time using the trapezoidal rule to obtain charge estimates.
* `detect_saturation` inspects the neighbourhood around the peak to tag clipped pulses:
    * Flags edge hits when the peak occurs near array boundaries.
    * Uses amplitude and flatness heuristics to identify flat-top regions (fractional coverage and consecutive samples).
* `analyze_waveform` ties everything together for a single trace (baseline removal, peak finding, saturation tagging, and multiple charge integrations).
* `analyze_waveforms_batch` iterates across waveforms, capturing summaries and any analysis exceptions.
* `summaries_to_dataframe` turns the summary objects into a pandas DataFrame for downstream use.

## Cell 6 – Plotting Utilities
* `plot_waveform` shows raw and baseline-corrected views of a single pulse, including the integration window for context.
* `plot_waveform_grid` arranges multiple corrected pulses in a grid for quick visual inspection of a set.

## Cell 7 – Analysis Driver
* Instantiates `RunConfig` (currently `Sample2` + `AmBe_20cmHDPE`) and loads either the full dataset or a capped subset when `max_waveforms` is set.
* Runs the load and analysis helpers, producing a `summary_df` with entry indices and filenames for traceability.
* Writes both CSV and pickle snapshots of the summary into `Data/Analysis_results` with sample/run in the filename.

## Cell 8 – Reporting and Charge Histogram
* Reports how many traces were marked saturated and previews their key metrics.
* Temporarily enables plotting to show example saturated waveforms and a grid of the first handful.
* Builds a histogram of the primary charge for non-saturated pulses, saves it alongside the summaries, and finally prints descriptive statistics.

## Cell 9 – Manual Plot Helper
* Quick convenience cell to re-plot a specific waveform (currently index 6) without re-running the entire analysis.

In [ ]:
csv_path = CONFIG.results_csv_path
if not csv_path.exists():
    print(f"Summary CSV not found at {csv_path}. Run the analysis cell to generate it first.")
else:
    loaded_df = pd.read_csv(csv_path)
    bool_columns = ["saturated", "baseline_spike", "post_peak_positive"]
    for column in bool_columns:
        if column in loaded_df.columns:
            loaded_df[column] = loaded_df[column].fillna(False).astype(bool)
        else:
            loaded_df[column] = False
            print(f"Column '{column}' missing from CSV; filled with False for downstream filters.")

    saturated_df = loaded_df[loaded_df["saturated"]]
    baseline_df = loaded_df[loaded_df["baseline_spike"]]
    post_peak_df = loaded_df[
        loaded_df["post_peak_positive"]
        & (~loaded_df["saturated"])
        & (~loaded_df["baseline_spike"])
    ]
    clean_df = loaded_df[
        (~loaded_df["saturated"])
        & (~loaded_df["baseline_spike"])
        & (~loaded_df["post_peak_positive"])
    ]

    category_frames_pre = {
        "All waveforms": loaded_df,
        "Saturated": saturated_df,
        "Baseline spike": baseline_df,
        "Post-peak (clean baseline)": post_peak_df,
        "Clean": clean_df,
    }
    pre_counts = {label: len(frame) for label, frame in category_frames_pre.items()}

    amplitude_cut = None
    if not saturated_df.empty and "amplitude_v" in saturated_df.columns:
        amplitude_cut = float(saturated_df["amplitude_v"].min())
        print(
            f"Applying amplitude cut >= {amplitude_cut:.4f} V derived from saturated subset (minimum amplitude)."
        )

    post_removed = clean_removed = 0
    if amplitude_cut is not None and "amplitude_v" in loaded_df.columns:
        post_before = len(post_peak_df)
        post_peak_df = post_peak_df[post_peak_df["amplitude_v"] < amplitude_cut]
        post_removed = post_before - len(post_peak_df)

        clean_before = len(clean_df)
        clean_df = clean_df[clean_df["amplitude_v"] < amplitude_cut]
        clean_removed = clean_before - len(clean_df)
        print(
            "Amplitude cut removed "
            f"{post_removed} post-peak and {clean_removed} clean entries with amplitude >= {amplitude_cut:.4f} V."
        )

    category_frames = {
        "All waveforms": loaded_df,
        "Saturated": saturated_df,
        "Baseline spike": baseline_df,
        "Post-peak (clean baseline)": post_peak_df,
        "Clean": clean_df,
    }
    post_counts = {label: len(frame) for label, frame in category_frames.items()}
    summary_table = pd.DataFrame(
        {
            "category": list(category_frames.keys()),
            "count_before": [pre_counts[label] for label in category_frames.keys()],
            "count_after": [post_counts[label] for label in category_frames.keys()],
        }
    )
    summary_table["removed"] = summary_table["count_before"] - summary_table["count_after"]
    display(summary_table)

    charge_columns = [column for column in loaded_df.columns if column.startswith("charge_")]
    if "charge_primary_v_s" in charge_columns:
        charge_column = "charge_primary_v_s"
    elif charge_columns:
        charge_column = charge_columns[0]
        print(f"Using {charge_column} for charge plots (primary charge column missing).")
    else:
        charge_column = None
        print("No charge columns detected in the summary CSV; charge plots will be skipped.")

    def plot_parameter_by_category(column: str, bins: int = 60, xlabel: str = "", title: str = "") -> None:
        available = {name: frame for name, frame in category_frames.items() if column in frame.columns and not frame.empty}
        if not available:
            print(f"No data available for {column} across the selected categories.")
            return
        fig, axes = plt.subplots(len(available), 1, figsize=(8, 3 * len(available)), sharex=True)
        axes = np.atleast_1d(axes)
        for axis, (label, frame) in zip(axes, available.items()):
            axis.hist(frame[column].dropna(), bins=bins, color="tab:blue", alpha=0.7, edgecolor="black")
            axis.set_ylabel("Count")
            axis.set_title(f"{label} (n={len(frame)})")
            axis.grid(alpha=0.3)
        axes[-1].set_xlabel(xlabel or column)
        if title:
            fig.suptitle(title, fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        # Attempt to save the plot to the CONFIG plots directory when available
        try:
            save_path = CONFIG.plot_path(f"{column}_by_category")
            save_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=150)
            print(f"Saved {column} distribution plot to {save_path}")
        except Exception as exc:
            print(f"Failed to save plot for {column}: {exc}")
        plt.show()

    plot_parameter_by_category(
        "amplitude_v",
        bins=80,
        xlabel="Amplitude [V]",
        title="Amplitude distribution by waveform category",
)

    if charge_column:
        plot_parameter_by_category(
            charge_column,
            bins=80,
            xlabel="Charge [V·s]",
            title=f"Charge distribution by waveform category ({charge_column})",
)
        clean_charge = clean_df[charge_column].dropna() if not clean_df.empty else pd.Series(dtype=float)
        post_charge = post_peak_df[charge_column].dropna() if not post_peak_df.empty else pd.Series(dtype=float)
        if clean_charge.empty and post_charge.empty:
            print("Clean and post-peak subsets are empty; skipping dedicated charge comparison plot.")
        else:
            bins = 80
            min_edge = min(clean_charge.min() if not clean_charge.empty else np.inf, post_charge.min() if not post_charge.empty else np.inf)
            max_edge = max(clean_charge.max() if not clean_charge.empty else -np.inf, post_charge.max() if not post_charge.empty else -np.inf)
            if not np.isfinite(min_edge) or not np.isfinite(max_edge) or min_edge == max_edge:
                print("Insufficient spread in charge data to build comparison histogram.")
            else:
                edges = np.linspace(min_edge, max_edge, bins + 1)
                plt.figure(figsize=(8, 4))
                if not clean_charge.empty:
                    plt.hist(clean_charge, bins=edges, histtype="step", linewidth=1.8, color="tab:green", label=f"Clean (n={len(clean_charge)})")
                if not post_charge.empty:
                    plt.hist(post_charge, bins=edges, histtype="step", linewidth=1.8, color="tab:orange", label=f"Post-peak (n={len(post_charge)})")
                plt.xlabel("Charge [V·s]")
                plt.ylabel("Count")
                plt.title("Charge distribution comparison: Clean vs Post-peak")
                plt.grid(alpha=0.3)
                plt.legend()
                plt.tight_layout()
                # save the comparison plot
                try:
                    save_path = CONFIG.plot_path("charge_clean_vs_post_peak")
                    save_path.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(save_path, dpi=150)
                    print(f"Saved charge comparison plot to {save_path}")
                except Exception as exc:
                    print(f"Failed to save charge comparison plot: {exc}")
                plt.show()

In [ ]:
clean_subset_path = CONFIG.subset_path("clean_waveforms")
if not clean_subset_path.exists():
    print(f"Clean subset pickle not found at {clean_subset_path}. Run the export cell to generate it.")
else:
    clean_subset_df = pd.read_pickle(clean_subset_path)
    if clean_subset_df.empty:
        print("Clean subset pickle is empty.")
    else:
        clean_subset_df = clean_subset_df.copy()
        initial_count = len(clean_subset_df)
        print(f"Loaded {initial_count} clean-waveform summaries from {clean_subset_path}.")
        bool_columns = ["saturated", "baseline_spike", "post_peak_positive"]
        for column in bool_columns:
            if column in clean_subset_df.columns:
                clean_subset_df[column] = clean_subset_df[column].fillna(False).astype(bool)
        if "post_peak_count" not in clean_subset_df.columns:
            clean_subset_df["post_peak_count"] = 0
        amplitude_cut = None
        summary_pkl_path = CONFIG.results_pkl_path
        summary_df_full = None
        if summary_pkl_path.exists():
            try:
                summary_df_full = pd.read_pickle(summary_pkl_path)
            except Exception as exc:
                print(f"Failed to load pulse summary pickle at {summary_pkl_path}: {exc}")
        else:
            print(f"Pulse summary pickle not found at {summary_pkl_path}.")
        if summary_df_full is not None:
            if "saturated" in summary_df_full.columns:
                summary_df_full["saturated"] = summary_df_full["saturated"].fillna(False).astype(bool)
            if {"saturated", "amplitude_v"}.issubset(summary_df_full.columns):
                saturated_amps = summary_df_full.loc[summary_df_full["saturated"], "amplitude_v"].dropna()
                if not saturated_amps.empty:
                    amplitude_cut = float(saturated_amps.min())
        if amplitude_cut is not None and "amplitude_v" in clean_subset_df.columns:
            pre_cut_count = len(clean_subset_df)
            clean_subset_df = clean_subset_df[clean_subset_df["amplitude_v"] < amplitude_cut].copy()
            removed_count = pre_cut_count - len(clean_subset_df)
            print(
                    f"Amplitude cut removed {removed_count} clean entries with amplitude >= {amplitude_cut:.4f} V.")
            print(f"{len(clean_subset_df)} entries remain after amplitude filtering.")
        elif amplitude_cut is None:
            print("Amplitude cut unavailable; skipping amplitude filtering.")
        else:
            print("Column 'amplitude_v' missing from clean subset; skipping amplitude filtering.")
        if clean_subset_df.empty:
            print("No clean entries remain after amplitude filtering.")
        else:
            multi_peak_mask = clean_subset_df.get("post_peak_count", 0) > 0
            multi_peak_df = clean_subset_df[multi_peak_mask].copy()
            if multi_peak_df.empty:
                print("No entries in the filtered clean subset report post-peak counts greater than zero.")
            else:
                print(
                    f"WARNING: Found {len(multi_peak_df)} filtered clean entries with post-peak_count > 0 (possible multi-peaks)."
                )
                display(multi_peak_df[[
                    "entry_index",
                    "file_name",
                    "amplitude_v",
                    "post_peak_positive",
                    "post_peak_count",
                    "post_peak_time_ns",
                    "post_peak_times_ns",
                ]].head(20))

In [ ]:
# Post-peak detailed analysis: amplitude and charge (integration -10 ns / +20 ns around post-peak)
if summary_df.empty:
    print("No summaries available — run the analysis driver (Cell 7) first.")
else:
    # build candidate mask: post-peak positive, not saturated, not baseline spike
    mask = summary_df.get("post_peak_positive", False) & (~summary_df.get("saturated", False)) & (~summary_df.get("baseline_spike", False))
    if amplitude_cut_to_refine_saturation is not None and "amplitude_v" in summary_df.columns:
        mask &= summary_df["amplitude_v"].fillna(-np.inf) < float(amplitude_cut_to_refine_saturation)
    candidates = summary_df[mask].copy()
    if candidates.empty:
        print("No post-peak candidates found under current filters.")
    else:
        records = []
        for _, row in candidates.iterrows():
            entry_index = int(row.get("entry_index", row.name))
            if entry_index < 0 or entry_index >= len(waveforms):
                continue
            wf = waveforms[entry_index]
            summ = summaries[entry_index]
            time_ns = wf.time_ns
            # baseline-corrected, polarity-aligned trace
            aligned = (wf.voltage_v - summ.baseline_v) * CONFIG.polarity_sign
            main_amp = float(summ.amplitude_v) if getattr(summ, "amplitude_v", None) is not None else float(np.nan)
            main_time = float(getattr(summ, 'peak_time_ns', float('nan')))
            # determine post-peak time and height
            post_time = float(summ.post_peak_time_ns) if np.isfinite(getattr(summ, "post_peak_time_ns", float("nan"))) else (float(summ.post_peak_times_ns[0]) if summ.post_peak_times_ns else float("nan"))
            post_height = float(summ.post_peak_height_v) if getattr(summ, "post_peak_height_v", None) is not None and np.isfinite(getattr(summ, "post_peak_height_v", float("nan"))) else float(np.nan)
            if not np.isfinite(post_height) and np.isfinite(post_time):
                # sample nearest
                idx = int(np.argmin(np.abs(time_ns - post_time))) if time_ns.size else None
                if idx is not None and 0 <= idx < len(aligned):
                    post_height = float(aligned[idx])
            main_charge = float(summ.charges_vns.get("primary", np.nan)) if getattr(summ, "charges_vns", None) is not None else float(np.nan)
            post_charge = float("nan")
            if np.isfinite(post_time):
                start_ns = post_time - 10.0
                end_ns = post_time + 20.0
                try:
                    post_charge = integrate_window(time_ns, aligned, start_ns, end_ns)
                except Exception:
                    post_charge = float("nan")
            records.append({
                "entry_index": entry_index,
                "file_name": row.get("file_name", ""),
                "main_amplitude_v": main_amp,
                "main_peak_time_ns": main_time,
                "post_amplitude_v": post_height,
                "main_charge_v_s": main_charge,
                "post_charge_v_s": post_charge,
                "post_peak_time_ns": post_time,
            })

        post_df = pd.DataFrame(records)
        if post_df.empty:
            print("No valid post-peak records were collected.")
        else:
            # save table
            out_csv = CONFIG.results_dir / f"{CONFIG.sample.replace(' ','_')}_{CONFIG.run_type.replace(' ','_')}_post_peak_summary.csv"
            out_csv.parent.mkdir(parents=True, exist_ok=True)
            post_df.to_csv(out_csv, index=False)
            print(f"Saved post-peak summary to {out_csv}")
            # compute delta time (post - main) in ns
            post_df["delta_time_ns"] = post_df["post_peak_time_ns"] - post_df["main_peak_time_ns"]
            # save augmented table
            out_csv2 = CONFIG.results_dir / f"{CONFIG.sample.replace(' ','_')}_{CONFIG.run_type.replace(' ','_')}_post_peak_summary_with_delta.csv"
            post_df.to_csv(out_csv2, index=False)
            print(f"Saved post-peak summary with delta to {out_csv2}")
            # 1) amplitude main vs post scatter
            plt.figure(figsize=(6,5))
            plt.scatter(post_df["main_amplitude_v"], post_df["post_amplitude_v"], s=24, alpha=0.7, color='tab:blue')
            mins = np.nanmin(post_df[["main_amplitude_v","post_amplitude_v"]].values)
            maxs = np.nanmax(post_df[["main_amplitude_v","post_amplitude_v"]].values)
            if np.isfinite(mins) and np.isfinite(maxs):
                plt.plot([mins, maxs],[mins, maxs], linestyle='--', color='gray')
            plt.xlabel('Main amplitude [V]')
            plt.ylabel('Post-peak amplitude [V]')
            plt.title('Main vs Post-peak amplitude (clean post-peak candidates)')
            plt.grid(alpha=0.3)
            try:
                ppath = CONFIG.plot_path('amplitude_main_vs_post_detailed')
                ppath.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(ppath, dpi=150)
                print(f"Saved amplitude scatter to {ppath}")
            except Exception as e:
                print('Failed saving amplitude scatter:', e)
            plt.show()
            # 2) main vs post charge
            good = post_df[post_df["main_charge_v_s"].notna() & post_df["post_charge_v_s"].notna()]
            if not good.empty:
                plt.figure(figsize=(6,5))
                plt.scatter(good["main_charge_v_s"], good["post_charge_v_s"], s=24, alpha=0.7, color='tab:green')
                plt.plot([good["main_charge_v_s"].min(), good["main_charge_v_s"].max()],[good["main_charge_v_s"].min(), good["main_charge_v_s"].max()], linestyle='--', color='gray')
                plt.xlabel('Main charge [V·s]')
                plt.ylabel('Post-peak charge [V·s]')
                plt.title('Main vs Post-peak charge (-10/+20 ns integration)')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('charge_main_vs_post_detailed')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved charge scatter to {ppath}")
                except Exception as e:
                    print('Failed saving charge scatter:', e)
                plt.show()
            else:
                print('Insufficient finite charges for main vs post scatter.')
            # 3) histograms of charges
            plt.figure(figsize=(8,4))
            if post_df["main_charge_v_s"].notna().any():
                plt.hist(post_df["main_charge_v_s"].dropna(), bins=60, alpha=0.6, label=f"Main (n={post_df['main_charge_v_s'].notna().sum()})", color='tab:green')
            if post_df["post_charge_v_s"].notna().any():
                plt.hist(post_df["post_charge_v_s"].dropna(), bins=60, alpha=0.6, label=f"Post (n={post_df['post_charge_v_s'].notna().sum()})", color='tab:orange')
            plt.xlabel('Charge [V·s]')
            plt.ylabel('Count')
            plt.title('Charge distributions: main vs post-peak (integrated -10/+20 ns)')
            plt.legend()
            plt.grid(alpha=0.3)
            try:
                ppath = CONFIG.plot_path('charge_hist_main_post_detailed')
                ppath.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(ppath, dpi=150)
                print(f"Saved charge histogram to {ppath}")
            except Exception as e:
                print('Failed saving charge histogram:', e)
            plt.show()
            # 4) delta time plots
            if "delta_time_ns" in post_df.columns and post_df["delta_time_ns"].notna().any():
                d = post_df["delta_time_ns"].dropna()
                plt.figure(figsize=(6,4))
                plt.hist(d, bins=60, color='tab:purple', alpha=0.8)
                plt.xlabel('Post - Main time [ns]')
                plt.ylabel('Count')
                plt.title('Delta time distribution (post - main)')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('delta_time_hist')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved delta time histogram to {ppath}")
                except Exception as e:
                    print('Failed saving delta time histogram:', e)
                plt.show()
                # scatter delta vs post amplitude
                plt.figure(figsize=(6,4))
                plt.scatter(post_df['delta_time_ns'], post_df['post_amplitude_v'], s=20, alpha=0.7, color='tab:orange')
                plt.xlabel('Delta time [ns]')
                plt.ylabel('Post-peak amplitude [V]')
                plt.title('Delta time vs post-peak amplitude')
                plt.grid(alpha=0.3)
                try:
                    ppath = CONFIG.plot_path('delta_vs_post_amplitude')
                    ppath.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(ppath, dpi=150)
                    print(f"Saved delta vs amplitude scatter to {ppath}")
                except Exception as e:
                    print('Failed saving delta vs amplitude scatter:', e)
                plt.show()
            else:
                print('No delta_time values available to plot.')

In [ ]:
# Aggregate primary charge from all summary files under Analysis_results and plot together
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

# try to import curve_fit for gaussian fitting; fall back gracefully if unavailable
try:
    from scipy.optimize import curve_fit
    curve_fit_available = True
except Exception:
    curve_fit_available = False

# simple gaussian model for fitting
def _gauss(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Determine results directory (prefer CONFIG if present)
try:
    results_root = Path(CONFIG.results_dir)
except Exception:
    results_root = Path.cwd() / 'Data' / 'Analysis_results'

# user-specified fit centers (primary-charge units) and extension targets for Compton-edge search
FIT_CENTERS = {
    'cs137': 0.38e-8,
    'na22': 0.25e-8,
}
# how far to extend the tail search for the right-hand crossing (absolute primary-charge units)
FIT_EXTEND_TO = {
    'cs137': 0.6e-8,
    'na22': 0.4e-8,
}
# fit half-width fraction (fraction of center) if not using absolute
FIT_HALF_WIDTH_FRAC = 0.20
# minimum absolute half-width to avoid tiny windows
FIT_HALF_WIDTH_MIN = 0.05e-8

if not results_root.exists():
    print(f"Results directory not found: {results_root}")
else:
    # collect candidate summary files (CSV and pickle)
    summary_files = sorted(results_root.glob('**/*summary*.csv')) + sorted(results_root.glob('**/*summary*.pkl'))
    # exclude any post-peak summary files
    summary_files = [f for f in summary_files if '_post_peak_summary' not in f.name]

    if not summary_files:
        print('No summary files found under', results_root)
    else:
        def extract_primary_from_series(s):
            if pd.isna(s):
                return np.nan
            if isinstance(s, (int, float, np.floating, np.integer)):
                return float(s)
            if isinstance(s, dict):
                return float(s.get('primary', np.nan))
            if isinstance(s, str):
                try:
                    obj = ast.literal_eval(s)
                    if isinstance(obj, dict):
                        return float(obj.get('primary', np.nan))
                except Exception:
                    try:
                        return float(s)
                    except Exception:
                        return np.nan
            return np.nan

        collected = {}
        for f in summary_files:
            try:
                if f.suffix.lower() == '.csv':
                    df = pd.read_csv(f)
                else:
                    df = pd.read_pickle(f)
            except Exception as e:
                print('Failed to read', f, ':', e)
                continue

            # Build a clean-mask: not saturated, not baseline_spike, not post_peak_positive (if those columns exist)
            clean_mask = pd.Series(True, index=df.index)
            if 'saturated' in df.columns:
                clean_mask &= ~df['saturated'].fillna(False)
            if 'baseline_spike' in df.columns:
                clean_mask &= ~df['baseline_spike'].fillna(False)
            if 'post_peak_positive' in df.columns:
                clean_mask &= ~df['post_peak_positive'].fillna(False)

            n_total = len(df)
            n_clean = int(clean_mask.sum())

            primary_series = None
            # prefer explicit 'primary' column
            if 'primary' in df.columns:
                primary_series = pd.to_numeric(df['primary'], errors='coerce')
            elif 'primary_charge' in df.columns:
                primary_series = pd.to_numeric(df['primary_charge'], errors='coerce')
            elif 'charges_vns' in df.columns:
                # charges_vns is expected to be dict-like; extract only 'primary' entries
                primary_series = df['charges_vns'].map(extract_primary_from_series).astype(float)
            elif 'charges' in df.columns:
                # if 'charges' column contains dict-like strings, extract primary
                primary_series = df['charges'].map(extract_primary_from_series).astype(float)
            else:
                # fallback: any column that contains 'charge' but exclude post-peak/post columns
                candidates = [c for c in df.columns if 'charge' in c.lower() and 'post' not in c.lower() and 'post_peak' not in c.lower()]
                # prefer columns with 'primary' in the name
                if candidates:
                    chosen = None
                    for c in candidates:
                        if 'primary' in c.lower():
                            chosen = c
                            break
                    if chosen is None:
                        chosen = candidates[0]
                    primary_series = pd.to_numeric(df[chosen], errors='coerce')

            if primary_series is None:
                print('No primary charge column found in', f.name)
                continue

            # restrict to clean events only
            primary_clean = primary_series[clean_mask].dropna().astype(float)
            if primary_clean.size == 0:
                print(f'No clean primary-charge values in {f.name} (clean count={n_clean} of {n_total})')
                continue

            arr = primary_clean.values
            label = f.stem
            collected[label] = arr
            print(f"Loaded {len(arr)} clean primary-charge values from {f.name} (label='{label}')")

        if not collected:
            print('No clean primary-charge arrays collected from any summary files.')
        else:
            # compute global bins (use robust percentiles)
            all_vals = np.concatenate(list(collected.values()))
            vmin, vmax = np.nanpercentile(all_vals, [1.0, 99.0])
            if vmin == vmax:
                vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
            bins = np.linspace(vmin, vmax, 80)

            # 1) produce and save one-by-one plots per sample (normalized to unity)
            for i, (label, arr) in enumerate(sorted(collected.items())):
                plt.figure(figsize=(7, 4))
                # plot normalized histogram (density=True so area integrates to 1)
                plt.hist(arr, bins=bins, color='tab:blue', alpha=0.8, density=True)
                med = np.median(arr)

                lbl_lower = label.lower()
                # if label contains 'cs137' or 'na22' attempt a gaussian fit to the distribution in a restricted window
                target_key = None
                if 'cs137' in lbl_lower:
                    target_key = 'cs137'
                elif 'na22' in lbl_lower or 'na-22' in lbl_lower:
                    target_key = 'na22'

                if target_key is not None and curve_fit_available:
                    fit_center = FIT_CENTERS.get(target_key, None)
                    extend_to = FIT_EXTEND_TO.get(target_key, None)
                    if fit_center is not None and np.isfinite(fit_center):
                        half_width = max(FIT_HALF_WIDTH_MIN, FIT_HALF_WIDTH_FRAC * abs(fit_center))
                        fit_min = fit_center - half_width
                        fit_max = fit_center + half_width
                        # compute density histogram and a smoothed version for edge finding
                        hist_d, edges_d = np.histogram(arr, bins=bins, density=True)
                        centers_d = 0.5 * (edges_d[:-1] + edges_d[1:])
                        # smooth histogram
                        sigma_bins = max(1, int(np.ceil(3.0)))
                        xk = np.arange(-4 * sigma_bins, 4 * sigma_bins + 1)
                        kernel = np.exp(-0.5 * (xk / float(sigma_bins)) ** 2)
                        kernel = kernel / kernel.sum()
                        hist_s = np.convolve(hist_d.astype(float), kernel, mode='same')

                        # Fit gaussian inside the narrow window to obtain mu/sigma if possible
                        mask_fit = (centers_d >= fit_min) & (centers_d <= fit_max)
                        fit_ok = False
                        popt = None
                        if mask_fit.sum() >= 5:
                            try:
                                x_fit = centers_d[mask_fit]
                                y_fit = hist_d[mask_fit]
                                p0 = [y_fit.max(), np.mean(x_fit), np.std(arr)]
                                popt, pcov = curve_fit(_gauss, x_fit, y_fit, p0=p0, maxfev=20000)
                                A_fit, mu_fit, sigma_fit = popt
                                fit_ok = True
                                xs = np.linspace(fit_min, fit_max, 400)
                                plt.plot(xs, _gauss(xs, *popt), color='tab:red', linewidth=1.6, label=f'Gauss fit μ={mu_fit:.3g}, σ={sigma_fit:.3g}')
                                print(f"{label}: Gaussian fit succeeded (μ={mu_fit:.6g}, σ={sigma_fit:.6g})")
                            except Exception as e:
                                print('Gaussian fit failed for', label, ':', e)
                        else:
                            print(f'Not enough bins in fit window for {label} (need >=5, got {mask_fit.sum()})')

                        # Determine peak and half-height using smoothed histogram (fall back to fit if needed)
                        peak_idx = int(np.nanargmax(hist_s))
                        peak_center = centers_d[peak_idx]
                        peak_height = float(hist_s[peak_idx])
                        half_height = 0.5 * peak_height

                        # Define right-side search limit: use FIT_EXTEND_TO if provided, else fit_max*2
                        if extend_to is None:
                            right_limit = max(fit_max * 2.0, centers_d[-1])
                        else:
                            right_limit = extend_to

                        # search for first index to the right of the peak where smoothed hist <= half_height and center <= right_limit
                        idxs_right = np.where((centers_d > peak_center) & (centers_d <= right_limit))[0]
                        edge_val = None
                        if idxs_right.size:
                            below = np.where(hist_s[idxs_right] <= half_height)[0]
                            if below.size:
                                first_below_idx = idxs_right[below[0]]
                                # linear interpolation between the bin just before and this bin
                                i0 = first_below_idx - 1
                                i1 = first_below_idx
                                if i0 >= 0:
                                    x0, y0 = centers_d[i0], hist_s[i0]
                                    x1, y1 = centers_d[i1], hist_s[i1]
                                    if y1 != y0:
                                        edge_val = x0 + (half_height - y0) * (x1 - x0) / (y1 - y0)
                                    else:
                                        edge_val = centers_d[i1]
                                else:
                                    edge_val = centers_d[first_below_idx]

                        # fallback: if fit is ok, use analytic half-width: mu + sigma*sqrt(2*ln(2))
                        if edge_val is None and fit_ok:
                            try:
                                from math import sqrt, log

                                edge_val = mu_fit + sigma_fit * sqrt(2.0 * log(2.0))
                                print(f'{label}: edge from gaussian analytic half-max = {edge_val:.6g}')
                            except Exception:
                                edge_val = None

                        # final fallback: use centers_d peak-of-derivative method
                        if edge_val is None:
                            deriv = np.gradient(hist_s, centers_d)
                            # find most negative derivative to the right of peak
                            right_deriv = deriv[(np.arange(len(centers_d)) > peak_idx) & (centers_d <= right_limit)]
                            if right_deriv.size:
                                min_idx_rel = int(np.argmin(right_deriv))
                                idx_global = np.arange(len(centers_d))[(np.arange(len(centers_d)) > peak_idx) & (centers_d <= right_limit)][min_idx_rel]
                                edge_val = centers_d[idx_global]
                                print(f'{label}: fallback edge from derivative at {edge_val:.6g}')
                        if edge_val is not None:
                            plt.axvline(edge_val, color='tab:orange', linestyle='-.', linewidth=1.2, label=f'edge={edge_val:.3g}')
                        else:
                            print(f'Could not determine edge for {label}')
                    else:
                        print('No predefined fit center for', label)
                else:
                    if target_key is not None and not curve_fit_available:
                        print('scipy.optimize.curve_fit not available; skipping Gaussian fit for', label)

                plt.axvline(med, color='black', linestyle='--', linewidth=1.0, alpha=0.9, label=f'median={med:.3g}')
                plt.xlabel('Primary charge [V·s]')
                plt.ylabel('Probability density')
                plt.title(f'Primary charge (clean subset): {label} (n={len(arr)})')
                plt.legend(loc='best')
                plt.grid(alpha=0.3)
                try:
                    out = CONFIG.plot_path(f'primary_charge_{label}_clean')
                    out.parent.mkdir(parents=True, exist_ok=True)
                    plt.savefig(out, dpi=150)
                    print('Saved', out)
                except Exception:
                    fallback = Path.cwd() / f'primary_charge_{label}_clean.png'
                    plt.savefig(fallback, dpi=150)
                    print('Saved', fallback)
                plt.show()
                plt.close()

            # 2) optional combined figure (overlay) for quick comparison (normalized)
            plt.figure(figsize=(10, 6))
            colors = plt.cm.tab20.colors
            for i, (label, arr) in enumerate(sorted(collected.items())):
                c = colors[i % len(colors)]
                plt.hist(arr, bins=bins, alpha=0.35, color=c, label=f"{label} (n={len(arr)})", density=True)
                med = np.median(arr)
                plt.axvline(med, color=c, linestyle='--', linewidth=1.0, alpha=0.9)

            plt.xlabel('Primary charge [V·s]')
            plt.ylabel('Probability density')
            plt.title('Primary charge distributions (clean subset, all summary files)')
            plt.legend(loc='best', fontsize='small')
            plt.grid(alpha=0.3)
            try:
                out = CONFIG.plot_path('primary_charge_all_samples_clean')
                out.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(out, dpi=150)
                print('Saved combined primary-charge figure to', out)
            except Exception:
                fallback = Path.cwd() / 'primary_charge_all_samples_clean.png'
                plt.savefig(fallback, dpi=150)
                print('Saved combined primary-charge figure to', fallback)
            plt.show()


In [ ]:
# Extract Compton edges for Cs137 and Na22 from clean primary-charge distributions
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast

# helper: get primary charge series from a DataFrame (same logic as earlier)
def extract_primary_series(df):
    def extract_primary_from_series(s):
        if pd.isna(s):
            return np.nan
        if isinstance(s, (int, float, np.floating, np.integer)):
            return float(s)
        if isinstance(s, dict):
            return float(s.get('primary', np.nan))
        if isinstance(s, str):
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, dict):
                    return float(obj.get('primary', np.nan))
            except Exception:
                try:
                    return float(s)
                except Exception:
                    return np.nan
        return np.nan

    if 'primary' in df.columns:
        return pd.to_numeric(df['primary'], errors='coerce')
    if 'primary_charge' in df.columns:
        return pd.to_numeric(df['primary_charge'], errors='coerce')
    if 'charges_vns' in df.columns:
        return df['charges_vns'].map(extract_primary_from_series).astype(float)
    if 'charges' in df.columns:
        return df['charges'].map(extract_primary_from_series).astype(float)
    # fallback
    candidates = [c for c in df.columns if 'charge' in c.lower() and 'post' not in c.lower()]
    if candidates:
        return pd.to_numeric(df[candidates[0]], errors='coerce')
    return None

# detect edge in array of primary charges
def detect_compton_edge(arr, nbins=600, smooth_sigma_bins=3, search_percentile_low=40):
    if arr.size < 10:
        return np.nan, None, None
    # robust bin range
    vmin, vmax = np.nanpercentile(arr, [1.0, 99.0])
    if vmin == vmax:
        vmin, vmax = np.nanmin(arr), np.nanmax(arr)
    bins = np.linspace(vmin, vmax, nbins)
    hist, edges = np.histogram(arr, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    # smooth histogram with gaussian kernel
    x = np.arange(-4 * smooth_sigma_bins, 4 * smooth_sigma_bins + 1)
    kernel = np.exp(-0.5 * (x / float(smooth_sigma_bins)) ** 2)
    kernel = kernel / kernel.sum()
    hist_s = np.convolve(hist.astype(float), kernel, mode='same')
    # derivative
    deriv = np.gradient(hist_s, centers)
    # search for most negative derivative above a lower percentile to avoid low-energy structure
    thr = np.percentile(centers, search_percentile_low)
    mask = centers >= thr
    if not np.any(mask):
        idx = int(np.argmin(deriv))
    else:
        idx = int(np.argmin(deriv[mask]))
        idx = np.arange(len(centers))[mask][idx]
    edge = centers[idx]
    return edge, (centers, hist, hist_s, deriv, idx)

# find and plot for Cs137 and Na22
try:
    results_root = Path(CONFIG.results_dir)
except Exception:
    results_root = Path.cwd() / 'Data' / 'Analysis_results'

samples = {'Cs137': [], 'Na22': []}
for f in sorted(results_root.glob('**/*summary*.csv')) + sorted(results_root.glob('**/*summary*.pkl')):
    name = f.name.lower()
    if '_post_peak_summary' in name:
        continue
    for key in list(samples.keys()):
        if key.lower() in name:
            samples[key].append(f)

# if none found by name, try stems containing sample
for key, files in samples.items():
    if not files:
        # try any file whose stem contains the short name
        cand = [f for f in results_root.rglob('*summary*') if key.lower() in f.stem.lower()]
        samples[key] = cand

found_any = False
results = {}
for key, files in samples.items():
    if not files:
        print(f'No summary files found for {key} under {results_root}')
        continue
    # aggregate clean primary charges from all files for this sample
    all_arrs = []
    for f in files:
        try:
            if f.suffix.lower() == '.csv':
                df = pd.read_csv(f)
            else:
                df = pd.read_pickle(f)
        except Exception as e:
            print('Failed to read', f, ':', e)
            continue
        # build clean mask if present
        clean_mask = pd.Series(True, index=df.index)
        if 'saturated' in df.columns:
            clean_mask &= ~df['saturated'].fillna(False)
        if 'baseline_spike' in df.columns:
            clean_mask &= ~df['baseline_spike'].fillna(False)
        if 'post_peak_positive' in df.columns:
            clean_mask &= ~df['post_peak_positive'].fillna(False)
        prim = extract_primary_series(df)
        if prim is None:
            continue
        arr = prim[clean_mask].dropna().astype(float).values
        if arr.size:
            all_arrs.append(arr)
    if not all_arrs:
        print(f'No clean primary charges found for {key}')
        continue
    arr = np.concatenate(all_arrs)
    found_any = True
    edge, details = detect_compton_edge(arr)
    centers, hist, hist_s, deriv, idx = details
    results[key] = {'edge': edge, 'arr': arr, 'centers': centers, 'hist': hist, 'hist_s': hist_s, 'deriv': deriv, 'idx': idx}
    print(f'{key}: detected edge at {edge:.6g} (units of primary charge)')
    # plot
    plt.figure(figsize=(8, 4))
    plt.plot(centers, hist, color='0.7', label='raw hist')
    plt.plot(centers, hist_s, color='tab:blue', label='smoothed hist')
    plt.axvline(edge, color='tab:red', linestyle='--', label=f'edge={edge:.3g}')
    plt.xlabel('Primary charge [V·s]')
    plt.ylabel('Count')
    plt.title(f'Compton edge detection for {key} (clean subset)')
    plt.legend()
    plt.grid(alpha=0.3)
    try:
        out = CONFIG.plot_path(f'compton_edge_{key}')
        out.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out, dpi=150)
        print('Saved', out)
    except Exception:
        plt.savefig(Path.cwd() / f'compton_edge_{key}.png', dpi=150)
    plt.show()

if not found_any:
    print('No sample data found for Cs137 or Na22. Make sure summary files exist under the Analysis_results folder.')
else:
    print('\nExtraction complete. Results dict contains detected edges in primary-charge units.')

In [ ]:
# Calibrate primary-charge → keVee using detected Compton edges
# Uses `results` produced by the Compton-edge extraction cell.
# Produces a linear calibration: Energy_keV = a * (primary_charge [V·s]) + b

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Known Compton-edge energies (keV electron-equivalent) — assumptions:
# - Cs137 (662 keV gamma): Compton edge ≈ 477 keV
# - Na22: prefer annihilation 511 keV Compton edge ≈ 341 keV
# If you'd like Na22 to use the 1274.5 keV gamma instead, set the value to ~1061 keV.
KNOWN_EDGE_KEV = {
    'Cs137': 477.0,
    'Na22': 341.0,
}

# Allow user override by editing this dict in the notebook if needed.

# Check results dict
try:
    results
except NameError:
    print('The `results` dictionary is not present in the kernel. Run the Compton-edge extraction cell first (the cell that produced `results`).')
else:
    measured = []
    energies = []
    used_keys = []
    for key, true_keV in KNOWN_EDGE_KEV.items():
        if key in results and results[key] is not None:
            edge = results[key].get('edge', None)
            if edge is None or (isinstance(edge, float) and np.isnan(edge)):
                print(f"Warning: measured edge for {key} is missing or NaN; skipping {key}.")
                continue
            measured.append(float(edge))
            energies.append(float(true_keV))
            used_keys.append(key)
        else:
            print(f"No results entry found for {key}; skipping.")

    if len(measured) < 2:
        print(f"Need at least two calibration points; found {len(measured)}. Cannot fit a linear calibration.")
        print("Options: (1) re-run Compton-edge extraction so `results` is populated; (2) add another source; (3) set a fixed scale manually.")
    else:
        measured = np.array(measured)
        energies = np.array(energies)
        # Fit linear calibration (keV = a * charge + b)
        a, b = np.polyfit(measured, energies, 1)
        print(f"Calibration: Energy_keV = a * charge + b")
        print(f"a = {a:.6e} keV / (V·s)")
        print(f"b = {b:.6e} keV")

        # Save calibration
        try:
            calib = {'a': float(a), 'b': float(b), 'points': {k: {'measured_edge': float(results[k]['edge']), 'energy_keV': KNOWN_EDGE_KEV[k]} for k in used_keys}}
        except Exception:
            calib = {'a': float(a), 'b': float(b), 'points': list(zip(used_keys, measured.tolist(), energies.tolist()))}
        try:
            out_path = Path(CONFIG.results_dir) / 'charge_to_keVee_calibration.json'
        except Exception:
            out_path = Path.cwd() / 'charge_to_keVee_calibration.json'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, 'w') as fh:
            json.dump(calib, fh, indent=2)
        print(f"Saved calibration to {out_path}")

        # Plot calibration points and fit
        plt.figure(figsize=(6,4))
        plt.scatter(measured, energies, color='tab:blue', s=80, zorder=5)
        xs = np.linspace(measured.min() * 0.8, measured.max() * 1.2, 200)
        plt.plot(xs, a * xs + b, color='tab:red', linewidth=1.6, label=f'fit: E = {a:.3e}·Q + {b:.3e}')
        for x, y, k in zip(measured, energies, used_keys):
            plt.annotate(f"{k}\n{y:.1f} keV\nmeas={x:.3g}", (x, y), textcoords='offset points', xytext=(6, -10))
        plt.xlabel('Primary charge [V·s]')
        plt.ylabel('Energy [keVee]')
        plt.title('Charge → keVee calibration')
        plt.grid(alpha=0.25)
        plt.legend()
        try:
            out_plot = CONFIG.plot_path('calibration_charge_to_keVee')
        except Exception:
            out_plot = Path.cwd() / 'calibration_charge_to_keVee.png'
        plt.savefig(out_plot, dpi=150)
        print(f"Saved calibration plot to {out_plot}")
        plt.show()

        # report residuals
        preds = a * measured + b
        resid = energies - preds
        for k, m, e, p, r in zip(used_keys, measured, energies, preds, resid):
            pct = 100.0 * r / e if e != 0 else np.nan
            print(f"{k}: measured_edge={m:.6g} → predicted_energy={p:.3f} keV (true {e:.3f} keV), residual {r:.3f} keV ({pct:.2f}%)")

        # Optional: convert the collected primary-charge histograms into keV and save an overlay (if collected exists)
        try:
            if 'collected' in globals() and isinstance(collected, dict) and collected:
                all_vals = np.concatenate(list(collected.values()))
                vmin, vmax = np.nanpercentile(all_vals, [1.0, 99.0])
                bins = np.linspace(vmin, vmax, 80)
                # plot converted histograms
                plt.figure(figsize=(10,6))
                colors = plt.cm.tab20.colors
                for i, (label, arr) in enumerate(sorted(collected.items())):
                    q = np.array(arr)
                    e_keV = a * q + b
                    plt.hist(e_keV, bins=80, alpha=0.35, color=colors[i % len(colors)], density=True, label=f"{label} (n={len(q)})")
                plt.xlabel('Energy [keVee]')
                plt.ylabel('Probability density')
                plt.title('Primary charge distributions converted to keVee (clean subset)')
                plt.legend(loc='best', fontsize='small')
                plt.grid(alpha=0.25)
                try:
                    out_all = CONFIG.plot_path('primary_charge_all_samples_clean_keVee')
                except Exception:
                    out_all = Path.cwd() / 'primary_charge_all_samples_clean_keVee.png'
                plt.savefig(out_all, dpi=150)
                print(f"Saved converted histogram overlay to {out_all}")
                plt.show()
        except Exception as e:
            print('Failed to produce converted histograms:', e)


In [ ]:
# Fit Cs137 and Na22 primary-charge spectra with an exponential + Gaussian model
# and extract a refined Compton-edge by locating the half-height crossing on the fitted model.

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

# Model: exponential background + gaussian peak
def exp_plus_gauss(x, C, k, A, mu, sigma):
    return C * np.exp(-k * x) + A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Helper to smooth histogram (simple Gaussian kernel)
def smooth_hist(hist, sigma_bins=3):
    xk = np.arange(-4 * sigma_bins, 4 * sigma_bins + 1)
    kernel = np.exp(-0.5 * (xk / float(sigma_bins)) ** 2)
    kernel = kernel / kernel.sum()
    return np.convolve(hist.astype(float), kernel, mode='same')

# targets (match keys used in `results` from earlier):
targets = ['Cs137', 'Na22']
extracted_edges = {}

for key in targets:
    if 'results' not in globals() or key not in results or results[key] is None:
        print(f"No data for {key} available in `results`; skipping.")
        continue
    arr = np.asarray(results[key].get('arr', []))
    if arr.size < 20:
        print(f"Not enough samples for {key} (n={arr.size}); skipping fit.")
        continue

    # build histogram
    nbins = 240
    vmin, vmax = np.nanpercentile(arr, [1.0, 99.0])
    if vmin == vmax:
        vmin, vmax = np.nanmin(arr), np.nanmax(arr)
    edges = np.linspace(vmin, vmax, nbins + 1)
    hist, _ = np.histogram(arr, bins=edges, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])

    # smooth for robust peak finding
    hist_s = smooth_hist(hist, sigma_bins=3)

    # initial guesses
    A0 = float(hist_s.max())
    mu0 = float(centers[np.argmax(hist_s)])
    sigma0 = max(1e-12, float(np.std(arr)) * 0.5)
    C0 = max(1e-12, float(hist_s.min()))
    # k initial guess: exponential scale inverse of mean positive values
    mean_q = max(1e-12, np.mean(arr[arr > 0]) if np.any(arr > 0) else np.mean(np.abs(arr)))
    k0 = 1.0 / mean_q if mean_q > 0 else 1.0
    p0 = [C0, k0, A0, mu0, sigma0]

    # bounds: keep positive where needed
    lower = [0.0, 0.0, 0.0, vmin, 1e-12]
    upper = [np.inf, np.inf, np.inf, vmax, (vmax - vmin) * 2.0]

    try:
        popt, pcov = curve_fit(exp_plus_gauss, centers, hist_s, p0=p0, bounds=(lower, upper), maxfev=20000)
        C_fit, k_fit, A_fit, mu_fit, sigma_fit = popt
        print(f"{key}: fit OK — mu={mu_fit:.6g}, sigma={sigma_fit:.6g}, A={A_fit:.6g}, C={C_fit:.6g}, k={k_fit:.6g}")
    except Exception as e:
        print(f"Fit failed for {key}: {e}")
        # still try to estimate edge from smoothed histogram peak
        peak_idx = int(np.nanargmax(hist_s))
        peak_center = centers[peak_idx]
        peak_height = float(hist_s[peak_idx])
        half_height = 0.5 * peak_height
        # search right of peak for half crossing
        idxs_right = np.where(centers > peak_center)[0]
        edge_val = None
        if idxs_right.size:
            below = np.where(hist_s[idxs_right] <= half_height)[0]
            if below.size:
                i1 = idxs_right[below[0]]
                i0 = i1 - 1
                if i0 >= 0:
                    x0, y0 = centers[i0], hist_s[i0]
                    x1, y1 = centers[i1], hist_s[i1]
                    if y1 != y0:
                        edge_val = x0 + (half_height - y0) * (x1 - x0) / (y1 - y0)
                    else:
                        edge_val = centers[i1]
        extracted_edges[key] = edge_val
        print(f"{key}: estimated edge (from smoothed hist half-height) = {edge_val}")
        continue

    # compute model on dense grid
    xs = np.linspace(vmin, max(vmax, mu_fit + max(3.0 * sigma_fit, (FIT_EXTEND_TO.get(key.lower(), FIT_EXTEND_TO.get(key, None)) or vmax))), 2000)
    model = exp_plus_gauss(xs, *popt)

    # find peak of model and half-height crossing to right
    peak_idx = int(np.nanargmax(model))
    peak_x = xs[peak_idx]
    peak_y = float(model[peak_idx])
    half_y = 0.5 * peak_y

    # define right_limit using FIT_EXTEND_TO if available
    right_limit = None
    try:
        if key.lower() in FIT_EXTEND_TO:
            right_limit = FIT_EXTEND_TO[key.lower()]
        elif key in FIT_EXTEND_TO:
            right_limit = FIT_EXTEND_TO[key]
    except Exception:
        right_limit = None
    if right_limit is None:
        right_limit = xs[-1]

    # search xs to the right of peak_x for first model <= half_y and <= right_limit
    mask_right = (xs > peak_x) & (xs <= right_limit)
    idxs = np.where(mask_right)[0]
    edge_val = None
    if idxs.size:
        rel = np.where(model[idxs] <= half_y)[0]
        if rel.size:
            i1 = idxs[rel[0]]
            i0 = i1 - 1
            if i0 >= 0:
                x0, y0 = xs[i0], model[i0]
                x1, y1 = xs[i1], model[i1]
                if y1 != y0:
                    edge_val = x0 + (half_y - y0) * (x1 - x0) / (y1 - y0)
                else:
                    edge_val = xs[i1]

    # fallback: use analytic from gaussian component if no crossing found
    if edge_val is None and A_fit > 0 and sigma_fit > 0:
        from math import sqrt, log
        try:
            # approximate half-max for gaussian component only: mu + sigma*sqrt(2*ln(2))
            edge_val = mu_fit + sigma_fit * sqrt(2.0 * log(2.0))
            print(f"{key}: fallback analytic gaussian half-max = {edge_val}")
        except Exception:
            edge_val = None

    extracted_edges[key] = edge_val

    # plotting
    plt.figure(figsize=(8,4))
    plt.plot(centers, hist, color='0.7', label='raw hist')
    plt.plot(centers, hist_s, color='tab:blue', label='smoothed hist')
    plt.plot(xs, model, color='tab:red', linewidth=1.6, label='fit: exp + gauss')
    plt.axvline(edge_val, color='tab:orange', linestyle='-.', linewidth=1.3, label=f'edge={edge_val:.3g}')
    plt.axvline(mu_fit, color='tab:green', linestyle='--', linewidth=1.0, label=f'mu={mu_fit:.3g}')
    plt.xlabel('Primary charge [V·s]')
    plt.ylabel('Probability density')
    plt.title(f'Exp + Gauss fit and extracted edge for {key}')
    plt.legend(loc='best')
    try:
        out = CONFIG.plot_path(f'exp_plus_gauss_{key}_edge')
    except Exception:
        out = Path.cwd() / f'exp_plus_gauss_{key}_edge.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150)
    print(f"Saved fit figure to {out}")
    plt.show()

print('\nRefined edges from fits:')
for k, v in extracted_edges.items():
    print(f"{k}: edge={v}")


In [ ]:
# Aggregate primary charge from all summary files under Analysis_results and plot together
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

# Determine results directory (prefer CONFIG if present)
try:
    results_root = Path(CONFIG.results_dir)
except Exception:
    results_root = Path.cwd() / 'Data' / 'Analysis_results'

if not results_root.exists():
    print(f"Results directory not found: {results_root}")
else:
    # collect candidate summary files (CSV and pickle)
    summary_files = sorted(results_root.glob('**/*summary*.csv')) + sorted(results_root.glob('**/*summary*.pkl'))
    if not summary_files:
        print('No summary files found under', results_root)
    else:
        def extract_primary_from_series(s):
            if pd.isna(s):
                return np.nan
            if isinstance(s, (int, float, np.floating, np.integer)):
                return float(s)
            if isinstance(s, dict):
                return float(s.get('primary', np.nan))
            if isinstance(s, str):
                try:
                    obj = ast.literal_eval(s)
                    if isinstance(obj, dict):
                        return float(obj.get('primary', np.nan))
                except Exception:
                    try:
                        return float(s)
                    except Exception:
                        return np.nan
            return np.nan

        collected = {}
        for f in summary_files:
            try:
                if f.suffix.lower() == '.csv':
                    df = pd.read_csv(f)
                else:
                    df = pd.read_pickle(f)
            except Exception as e:
                print('Failed to read', f, ':', e)
                continue

            primary_series = None
            # common storage patterns
            if 'primary' in df.columns:
                primary_series = pd.to_numeric(df['primary'], errors='coerce')
            elif 'primary_charge' in df.columns:
                primary_series = pd.to_numeric(df['primary_charge'], errors='coerce')
            elif 'charges_vns' in df.columns:
                primary_series = df['charges_vns'].map(extract_primary_from_series).astype(float)
            elif 'charges' in df.columns:
                primary_series = df['charges'].map(extract_primary_from_series).astype(float)
            else:
                # fallback: any column with 'charge' in the name
                candidates = [c for c in df.columns if 'charge' in c.lower()]
                if candidates:
                    primary_series = pd.to_numeric(df[candidates[0]], errors='coerce')

            if primary_series is None:
                print('No primary charge column found in', f.name)
                continue

            arr = primary_series.dropna().values.astype(float)
            if arr.size == 0:
                print('No finite primary charge values in', f.name)
                continue

            label = f.stem
            collected[label] = arr
            print(f"Loaded {len(arr)} primary-charge values from {f.name} (label='{label}')")

        if not collected:
            print('No primary-charge arrays collected from any summary files.')
        else:
            plt.figure(figsize=(10, 6))
            colors = plt.cm.tab20.colors
            # determine global bins across all collections: use percentiles to avoid long tails
            all_vals = np.concatenate(list(collected.values()))
            vmin, vmax = np.nanpercentile(all_vals, [1.0, 99.0])
            if vmin == vmax:
                vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
            bins = np.linspace(vmin, vmax, 80)

            for i, (label, arr) in enumerate(collected.items()):
                c = colors[i % len(colors)]
                plt.hist(arr, bins=bins, alpha=0.4, color=c, label=f"{label} (n={len(arr)})")
                med = np.median(arr)
                plt.axvline(med, color=c, linestyle='--', linewidth=1.0, alpha=0.9)

            plt.xlabel('Primary charge [V·s]')
            plt.ylabel('Count')
            plt.title('Primary charge distributions (all summary files)')
            plt.legend(loc='best', fontsize='small')
            plt.grid(alpha=0.3)

            # save figure via CONFIG if available
            try:
                out = CONFIG.plot_path('primary_charge_all_samples')
                out.parent.mkdir(parents=True, exist_ok=True)
                plt.savefig(out, dpi=150)
                print('Saved combined primary-charge figure to', out)
            except Exception:
                fallback = Path.cwd() / 'primary_charge_all_samples.png'
                plt.savefig(fallback, dpi=150)
                print('Saved combined primary-charge figure to', fallback)

            plt.show()

In [ ]:
# Overlay Na22, Cs137 and Am241 primary-charge spectra (clean subset)
# Searches the `collected` dict assembled earlier and overlays normalized histograms.

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Choose normalization: 'peak' -> max(bin) == 1.0, 'area' -> integral == 1.0
NORMALIZE_METHOD = 'peak'  # options: 'peak' or 'area'

# helper to find best matching labels in `collected`
def find_label(collected, keywords):
    found = {}
    keys = list(collected.keys()) if isinstance(collected, dict) else []
    for kw in keywords:
        kw_low = kw.lower()
        # prefer exact contains
        candidates = [k for k in keys if kw_low in k.lower()]
        if candidates:
            # if multiple, pick the one with most counts
            candidates = sorted(candidates, key=lambda k: len(collected.get(k,[])), reverse=True)
            found[kw] = candidates[0]
        else:
            found[kw] = None
    return found

# keywords to overlay
keywords = ['na22', 'cs137', 'am241']
labels_map = {}
if 'collected' in globals() and isinstance(collected, dict) and collected:
    labels_map = find_label(collected, keywords)
else:
    print('`collected` dict not found or empty. Run the aggregation cell first to populate it.')

# assemble arrays for plotting
plot_items = []
for kw in keywords:
    lbl = labels_map.get(kw)
    if lbl is None:
        print(f'No match found in `collected` for {kw}')
        continue
    arr = np.asarray(collected.get(lbl, []))
    if arr.size == 0:
        print(f'Found label {lbl} for {kw} but it contains no entries; skipping.')
        continue
    plot_items.append((kw.upper(), lbl, arr))

if not plot_items:
    print('No spectra available to overlay.')
else:
    # global bins from concatenated arrays
    all_vals = np.concatenate([it[2] for it in plot_items])
    vmin, vmax = np.nanpercentile(all_vals, [0.5, 99.5])
    if vmin == vmax:
        vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
    bins = np.linspace(vmin, vmax, 120)

    plt.figure(figsize=(10,6))
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    for i, (kwlabel, lbl, arr) in enumerate(plot_items):
        counts, edges = np.histogram(arr, bins=bins)
        centers = 0.5 * (edges[:-1] + edges[1:])
        if NORMALIZE_METHOD == 'peak':
            denom = counts.max() if counts.max() > 0 else 1.0
            norm_counts = counts.astype(float) / float(denom)
        else:  # area
            area = np.trapz(counts.astype(float), centers)
            denom = area if area > 0 else 1.0
            norm_counts = counts.astype(float) / float(denom)
        plt.step(centers, norm_counts, where='mid', color=colors[i%len(colors)], linewidth=1.8, label=f'{kwlabel} ({lbl}, n={len(arr)})')
        # mark median on normalized scale
        med = np.median(arr)
        # find bin index for median to place a marker at normalized height
        med_idx = np.searchsorted(centers, med)
        med_idx = min(max(med_idx, 0), len(norm_counts)-1)
        plt.plot([med], [norm_counts[med_idx]], marker='v', color=colors[i%len(colors)], markersize=6)

    plt.xlabel('Primary charge [V·s]')
    plt.ylabel('Normalized counts (unity: {})'.format('peak' if NORMALIZE_METHOD=='peak' else 'area'))
    plt.title('Overlay: Na22, Cs137, Am241 (clean subset) — normalized to unity')
    plt.legend(loc='best')
    plt.grid(alpha=0.25)
    try:
        out = CONFIG.plot_path('overlay_na22_cs137_am241_normalized')
    except Exception:
        out = Path.cwd() / 'overlay_na22_cs137_am241_normalized.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150)
    print(f'Saved overlay plot to {out}')
    plt.show()

    # If calibration exists, attempt to convert to keVee and plot overlay in energy units
    try:
        calib_path = Path(CONFIG.results_dir) / 'charge_to_keVee_calibration.json'
        if calib_path.exists():
            with open(calib_path, 'r') as fh:
                calib = json.load(fh)
            a = float(calib.get('a'))
            b = float(calib.get('b'))
            plt.figure(figsize=(10,6))
            for i, (kwlabel, lbl, arr) in enumerate(plot_items):
                e_keV = a * arr + b
                counts, edges = np.histogram(e_keV, bins=80)
                centers = 0.5 * (edges[:-1] + edges[1:])
                if NORMALIZE_METHOD == 'peak':
                    denom = counts.max() if counts.max() > 0 else 1.0
                    norm_counts = counts.astype(float) / float(denom)
                else:
                    area = np.trapz(counts.astype(float), centers)
                    denom = area if area > 0 else 1.0
                    norm_counts = counts.astype(float) / float(denom)
                plt.step(centers, norm_counts, where='mid', color=colors[i%len(colors)], linewidth=1.6, label=f'{kwlabel} (n={len(arr)})')
            plt.xlabel('Energy [keVee]')
            plt.ylabel('Normalized counts (unity: {})'.format('peak' if NORMALIZE_METHOD=='peak' else 'area'))
            plt.title('Overlay converted to keVee (using saved calibration) — normalized to unity')
            plt.legend(loc='best')
            plt.grid(alpha=0.25)
            try:
                out2 = CONFIG.plot_path('overlay_na22_cs137_am241_keVee_normalized')
            except Exception:
                out2 = Path.cwd() / 'overlay_na22_cs137_am241_keVee_normalized.png'
            plt.savefig(out2, dpi=150)
            print(f'Saved converted overlay to {out2}')
            plt.show()
        else:
            print('Calibration file not found; skipping keVee-converted overlay.')
    except Exception as e:
        print('Failed to produce keVee overlay:', e)


In [ ]:
# Overlay Na22, Cs137 and Am241 primary-charge spectra (clean subset)
# Searches the `collected` dict assembled earlier and overlays normalized histograms.

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Choose normalization: 'peak' -> max(bin) == 1.0, 'area' -> integral == 1.0
NORMALIZE_METHOD = 'peak'  # options: 'peak' or 'area'

# helper to find best matching labels in `collected`
def find_label(collected, keywords):
    found = {}
    keys = list(collected.keys()) if isinstance(collected, dict) else []
    for kw in keywords:
        kw_low = kw.lower()
        # prefer exact contains
        candidates = [k for k in keys if kw_low in k.lower()]
        if candidates:
            # if multiple, pick the one with most counts
            candidates = sorted(candidates, key=lambda k: len(collected.get(k,[])), reverse=True)
            found[kw] = candidates[0]
        else:
            found[kw] = None
    return found

# keywords to overlay
keywords = ['na22', 'cs137', 'am241','ambe', 'ambe_20cmHDPE','am241_gamma']
labels_map = {}
if 'collected' in globals() and isinstance(collected, dict) and collected:
    labels_map = find_label(collected, keywords)
else:
    print('`collected` dict not found or empty. Run the aggregation cell first to populate it.')

# assemble arrays for plotting
plot_items = []
for kw in keywords:
    lbl = labels_map.get(kw)
    if lbl is None:
        print(f'No match found in `collected` for {kw}')
        continue
    arr = np.asarray(collected.get(lbl, []))
    if arr.size == 0:
        print(f'Found label {lbl} for {kw} but it contains no entries; skipping.')
        continue
    plot_items.append((kw.upper(), lbl, arr))

if not plot_items:
    print('No spectra available to overlay.')
else:
    # global bins from concatenated arrays
    all_vals = np.concatenate([it[2] for it in plot_items])
    vmin, vmax = np.nanpercentile(all_vals, [0.5, 99.5])
    if vmin == vmax:
        vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
    bins = np.linspace(vmin, vmax, 120)

    plt.figure(figsize=(10,6))
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
    for i, (kwlabel, lbl, arr) in enumerate(plot_items):
        counts, edges = np.histogram(arr, bins=bins)
        centers = 0.5 * (edges[:-1] + edges[1:])
        if NORMALIZE_METHOD == 'peak':
            denom = counts.max() if counts.max() > 0 else 1.0
            norm_counts = counts.astype(float) / float(denom)
        else:  # area
            area = np.trapz(counts.astype(float), centers)
            denom = area if area > 0 else 1.0
            norm_counts = counts.astype(float) / float(denom)
        plt.step(centers, norm_counts, where='mid', color=colors[i%len(colors)], linewidth=1.8, label=f'{kwlabel} ({lbl}, n={len(arr)})')
        # mark median on normalized scale
        med = np.median(arr)
        # find bin index for median to place a marker at normalized height
        med_idx = np.searchsorted(centers, med)
        med_idx = min(max(med_idx, 0), len(norm_counts)-1)
        plt.plot([med], [norm_counts[med_idx]], marker='v', color=colors[i%len(colors)], markersize=6)

    plt.xlabel('Primary charge [V·s]')
    plt.ylabel('Normalized counts (unity: {})'.format('peak' if NORMALIZE_METHOD=='peak' else 'area'))
    plt.title('Overlay: Na22, Cs137, Am241 (clean subset) — normalized to unity')
    plt.legend(loc='best')
    plt.grid(alpha=0.25)
    try:
        out = CONFIG.plot_path('overlay_na22_cs137_am241_normalized')
    except Exception:
        out = Path.cwd() / 'overlay_na22_cs137_am241_normalized.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150)
    print(f'Saved overlay plot to {out}')
    plt.show()

    